# 01. UGVNet training workflow

**This cell covers:** What the notebook trains, how the data should be arranged, and the order to run it.

This Kaggle edition trains **UGVNet (Unified Global Vision Network)** with parallel **EfficientNetV2-S + ConvNeXt-Tiny** backbones, adaptive gated feature fusion, and global-attention refinement. Run the numbered cells from top to bottom.

> **Strong recommendation:** Use a tuned, deduplicated dataset with independent `train`, `validation`, and `test` folders. Images from the same patient, capture session, or augmented source must stay in one split only.

Attach a dataset with **Add Input**, point to a mounted input path, or provide a public dataset handle.

Before training, Cell 08 scans and decodes the entire dataset, shows the findings, and stops the run when the strict audit finds a critical problem.


## 02. Check the runtime

**This cell covers:** Python, PyTorch, TorchVision, and GPU availability in the current Kaggle session.


In [ ]:
# 02. Runtime check
import os
import platform
import sys
from pathlib import Path

import torch
import torchvision

print(f"Python:       {sys.version.split()[0]}")
print(f"Platform:     {platform.platform()}")
print(f"PyTorch:      {torch.__version__}")
print(f"TorchVision:  {torchvision.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:   {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")
else:
    print("WARNING: Enable a GPU accelerator before training this dual-backbone model.")

## 03. Imports

**This cell covers:** All standard-library, data-processing, plotting, PyTorch, TorchVision, and image imports used by later cells.

In [ ]:
# 03. Imports
import json
import math
import random
import shutil
import time
from collections import Counter
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import HTML, display
from PIL import Image
from torch import nn

try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    import subprocess

    print("Installing TensorBoard for this runtime...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tensorboard>=2.16",
    ])
    from torch.utils.tensorboard import SummaryWriter

# The trace is still recorded without this optional UI plugin. Installation is
# best-effort so an offline Kaggle runtime never blocks model training.
import importlib.util
import subprocess

TENSORBOARD_PROFILER_AVAILABLE = (
    importlib.util.find_spec("tensorboard_plugin_profile") is not None
)
if not TENSORBOARD_PROFILER_AVAILABLE:
    try:
        print("Installing the optional TensorBoard Profiler plugin...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "tensorboard-plugin-profile>=2.16",
        ])
        TENSORBOARD_PROFILER_AVAILABLE = True
    except Exception as error:  # noqa: BLE001
        print(
            "Profiler UI plugin is unavailable; training and all other "
            f"TensorBoard dashboards will continue normally ({error})."
        )

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.profiler import (
    ProfilerActivity,
    profile,
    schedule,
    tensorboard_trace_handler,
)
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import (
    ConvNeXt_Tiny_Weights,
    EfficientNet_V2_S_Weights,
    convnext_tiny,
    efficientnet_v2_s,
)

import logging
# Install the Inter font for matplotlib so charts render with Inter instead
# of falling back to DejaVu Sans and spamming font-not-found warnings.
import urllib.request
import matplotlib as _mpl
import matplotlib.font_manager as _fm
from pathlib import Path as _Path

_inter_url = (
    "https://github.com/google/fonts/raw/main/ofl/inter/"
    "Inter%5Bopsz%2Cwght%5D.ttf"
)
_inter_dest = _Path(_mpl.get_cachedir()).parent / "fonts" / "Inter.ttf"
if not _inter_dest.exists():
    _inter_dest.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(_inter_url, _inter_dest)
    _fm.fontManager.addfont(str(_inter_dest))
    _fm._load_fontmanager(try_read_cache=False)
    print(f"Installed Inter font: {_inter_dest}")
else:
    _fm.fontManager.addfont(str(_inter_dest))
    print("Inter font already installed.")

plt.style.use("seaborn-v0_8-whitegrid")

display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

:root {
    color-scheme: light dark;
    --ugv-text: #1e293b;
    --ugv-muted: #64748b;
    --ugv-surface: #ffffff;
    --ugv-surface-alt: #f8fafc;
    --ugv-hover: #f1f5f9;
    --ugv-line: #e2e8f0;
    --ugv-note-bg: linear-gradient(135deg, #eff6ff 0%, #f5f3ff 100%);
    --ugv-shadow: rgba(148, 163, 184, 0.12);
    --ugv-header-bg: #e0ecff;
    --ugv-header-text: #1e3a5f;
    --ugv-accent: #6366f1;
    --ugv-train: #bfdbfe;
    --ugv-val: #bbf7d0;
    --ugv-test: #fecaca;
    --ugv-total: #e9d5ff;
}

@media (prefers-color-scheme: dark) {
    :root {
        --ugv-text: #f1f5f9;
        --ugv-muted: #94a3b8;
        --ugv-surface: #0f172a;
        --ugv-surface-alt: #1e293b;
        --ugv-hover: #334155;
        --ugv-line: #334155;
        --ugv-note-bg: linear-gradient(135deg, #0f172a 0%, #1e1b4b 100%);
        --ugv-shadow: rgba(0, 0, 0, 0.35);
        --ugv-header-bg: #1e3a5f;
        --ugv-header-text: #e0ecff;
        --ugv-accent: #818cf8;
        --ugv-train: #1e3a5f;
        --ugv-val: #14532d;
        --ugv-test: #7f1d1d;
        --ugv-total: #3b0764;
    }
}

html[theme="dark"],
body[theme="dark"],
body[data-theme="dark"] {
    --ugv-text: #f1f5f9;
    --ugv-muted: #94a3b8;
    --ugv-surface: #0f172a;
    --ugv-surface-alt: #1e293b;
    --ugv-hover: #334155;
    --ugv-line: #334155;
    --ugv-note-bg: linear-gradient(135deg, #0f172a 0%, #1e1b4b 100%);
    --ugv-shadow: rgba(0, 0, 0, 0.35);
    --ugv-header-bg: #1e3a5f;
    --ugv-header-text: #e0ecff;
    --ugv-accent: #818cf8;
    --ugv-train: #1e3a5f;
    --ugv-val: #14532d;
    --ugv-test: #7f1d1d;
    --ugv-total: #3b0764;
}

.ugvnet-note {
    font-family: 'Inter', ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;
    color: var(--ugv-text) !important;
    background: var(--ugv-note-bg);
    border: 1px solid var(--ugv-line);
    border-left: 4px solid var(--ugv-accent);
    border-radius: 10px;
    margin: 12px 0 18px;
    padding: 14px 18px;
}

table.ugvnet-dataframe {
    background: var(--ugv-surface) !important;
    border: 0 !important;
    border-collapse: separate !important;
    border-spacing: 0;
    border-radius: 12px;
    box-shadow: 0 4px 16px var(--ugv-shadow);
    color: var(--ugv-text) !important;
    font-family: 'Inter', ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;
    overflow: hidden;
    width: 100%;
}

table.ugvnet-dataframe caption {
    caption-side: top;
    color: var(--ugv-text) !important;
    font-family: 'Inter', ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;
    font-size: 17px;
    font-weight: 700;
    letter-spacing: -0.01em;
    padding: 10px 2px 12px;
    text-align: left;
}

table.ugvnet-dataframe thead th {
    background: var(--ugv-header-bg) !important;
    border: 0 !important;
    color: var(--ugv-header-text) !important;
    font-family: 'Inter', ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;
    font-size: 12.5px;
    font-weight: 600;
    letter-spacing: 0.03em;
    padding: 10px 14px !important;
    text-align: left !important;
    text-transform: uppercase;
}

table.ugvnet-dataframe tbody td {
    background: var(--ugv-surface) !important;
    border: 0 !important;
    border-bottom: 1px solid var(--ugv-line) !important;
    color: var(--ugv-text) !important;
    font-family: 'Inter', ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;
    font-size: 13px;
    font-weight: 400;
    padding: 9px 14px !important;
    text-align: left !important;
}

table.ugvnet-dataframe tbody tr:nth-child(even) td {
    background: var(--ugv-surface-alt) !important;
}

table.ugvnet-dataframe tbody tr:hover td {
    background: var(--ugv-hover) !important;
    transition: background 0.15s ease;
}

.ugvnet-metric-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 12px;
    margin: 14px 0 20px;
}

.ugvnet-metric-card {
    background: var(--ugv-surface);
    border: 1px solid var(--ugv-line);
    border-radius: 14px;
    box-shadow: 0 4px 14px var(--ugv-shadow);
    color: var(--ugv-text) !important;
    font-family: 'Inter', ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;
    padding: 16px;
}

.ugvnet-metric-label {
    color: var(--ugv-muted) !important;
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 0.03em;
    text-transform: uppercase;
}

.ugvnet-metric-value {
    color: var(--ugv-text) !important;
    font-size: 27px;
    font-weight: 800;
    margin: 6px 0 2px;
}

.ugvnet-metric-detail {
    color: var(--ugv-muted) !important;
    font-size: 12px;
}

/* Text-based output styling */
.output_text pre, .output_subarea pre, .output_stream pre, div[data-mime-type="text/plain"] pre {
    font-family: 'Inter', ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif !important;
    font-size: 13px !important;
    line-height: 1.6 !important;
    color: var(--ugv-text) !important;
}

</style>
"""))


def show_dataframe(dataframe, title):
    """Display a DataFrame with light pastel styling and proper fonts."""
    if dataframe.empty:
        display(HTML(
            f'<div class="ugvnet-note"><strong>{title}</strong><br>'
            'No rows to show.</div>'
        ))
        return

    numeric_cols = dataframe.select_dtypes(include="number").columns.tolist()
    col_format = {}
    for col in dataframe.columns:
        if col in numeric_cols:
            col_format[col] = "{:,.0f}"
        else:
            col_format[col] = "{}"

    styled = (
        dataframe.style
        .hide(axis="index")
        .set_caption(title)
        .set_table_attributes('class="ugvnet-dataframe"')
        .format(col_format, na_rep="-")
    )

    # Apply light pastel column highlights for known split columns.
    pastel_map = {
        "train": "background-color: var(--ugv-train)",
        "Train": "background-color: var(--ugv-train)",
        "validation": "background-color: var(--ugv-val)",
        "Validation": "background-color: var(--ugv-val)",
        "test": "background-color: var(--ugv-test)",
        "Test": "background-color: var(--ugv-test)",
        "total": "background-color: var(--ugv-total)",
        "Total": "background-color: var(--ugv-total)",
    }
    for col_name, css in pastel_map.items():
        if col_name in dataframe.columns:
            styled = styled.map(lambda _: css, subset=[col_name])

    display(styled)


def show_accuracy_summary(rows):
    """Show final accuracy cards with adaptive light/dark text colors."""
    cards = []
    for label, value, detail in rows:
        cards.append(
            '<div class="ugvnet-metric-card">'
            f'<div class="ugvnet-metric-label">{label}</div>'
            f'<div class="ugvnet-metric-value">{value:.2%}</div>'
            f'<div class="ugvnet-metric-detail">Decimal: {value:.4f} \u2022 {detail}</div>'
            '</div>'
        )
    display(HTML('<div class="ugvnet-metric-grid">' + ''.join(cards) + '</div>'))



## 04. Experiment settings

**This cell covers:** Kaggle input settings, output paths, image size, batch size, optimizer values, model dimensions, automatic small/large-dataset behavior, and reproducibility.

Attach the dataset with **Add Input**, then copy its mounted path into `KAGGLE_INPUT_PATH`. The default writes directly to native folders under `/kaggle/working/ugvnet`; no extra platform subfolder is added.


In [ ]:
# 04. Experiment settings
PLATFORM = "kaggle"

# Native Kaggle working-storage layout.
PROJECT_OUTPUT_ROOT = Path("/kaggle/working/ugvnet")
RESULTS_DIR = PROJECT_OUTPUT_ROOT / "results"
BEST_MODELS_DIR = PROJECT_OUTPUT_ROOT / "models" / "best"
CHECKPOINTS_DIR = PROJECT_OUTPUT_ROOT / "models" / "checkpoints"
REPORTS_DIR = PROJECT_OUTPUT_ROOT / "reports"
CACHE_DIR = PROJECT_OUTPUT_ROOT / "cache"
TORCH_HOME_DIR = CACHE_DIR / "torch"
TENSORBOARD_ROOT = PROJECT_OUTPUT_ROOT / "tensorboard"

# Choose: kaggle_input, kaggle_handle, or local_path.
DATA_SOURCE = "kaggle_input"
KAGGLE_INPUT_PATH = "/kaggle/input/your-dataset"
KAGGLE_DATASET_HANDLE = ""
LOCAL_DATASET_PATH = CACHE_DIR / "dataset"

SEED = 42
BATCH_SIZE = 16
EVAL_BATCH_MULTIPLIER = 2
PREFETCH_FACTOR = 4

# The automatic profile keeps the full hybrid model but reduces expensive
# spatial work once the training split reaches a practical large-data size.
SMALL_DATASET_IMAGE_SIZE = 300
LARGE_DATASET_IMAGE_SIZE = 224
SMALL_DATASET_FUSION_CHANNELS = 384
LARGE_DATASET_FUSION_CHANNELS = 256
SMALL_DATASET_FUSION_DEPTH = 2
LARGE_DATASET_FUSION_DEPTH = 1
NUM_WORKERS = 4
EPOCHS = 60
PATIENCE = 12

BACKBONE_LR = 3e-5
FUSION_LR = 3e-4
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.10
GRADIENT_CLIP = 1.0

PRETRAINED = True
ATTENTION_HEADS = 8
DROPOUT = 0.20

TRAINING_MODE = "auto"
SMALL_DATASET_THRESHOLD = 8_000
SMALL_DATASET_FREEZE_EPOCHS = 5
LARGE_DATASET_FREEZE_EPOCHS = 2
CLASS_BALANCE = "auto"
BALANCED_SAMPLER_STRATEGY = "effective_number"
EFFECTIVE_NUMBER_BETA = "auto"
MINORITY_AUGMENTATION = "auto"
MINORITY_AUGMENTATION_MAX_FRACTION = 0.50
LOSS_FUNCTION = "auto"
FOCAL_LOSS_IMBALANCE_THRESHOLD = 3.0
FOCAL_GAMMA = 1.5
# Leave empty for a new run; set to ugvnet_hybrid_last.pt to continue a run.
RESUME_CHECKPOINT_PATH = ""

# A large gap never rejects an epoch. It only tunes the next epoch.
GENERALIZATION_GAP_THRESHOLD = 0.10
GAP_ADAPTATION_START_EPOCH = 3
GAP_ADAPTATION_PATIENCE = 2
GAP_ADAPTATION_LR_FACTOR = 0.70
GAP_ADAPTATION_DROPOUT_STEP = 0.03
MAX_ADAPTIVE_DROPOUT = 0.40
GAP_ADAPTATION_LABEL_SMOOTHING_STEP = 0.02
MAX_ADAPTIVE_LABEL_SMOOTHING = 0.20
GAP_ADAPTATION_FOCAL_GAMMA_STEP = 0.25
MIN_ADAPTIVE_FOCAL_GAMMA = 0.50
USE_AMP = True
USE_FUSED_ADAMW = True
AUDIT_POLICY = "strict"
AUDIT_WORKERS = 8

# TensorBoard uses sampled diagnostics to keep the training overhead small.
TENSORBOARD_ENABLED = True
TENSORBOARD_RUN_NAME = ""  # Leave empty for a timestamped run name.
TENSORBOARD_FLUSH_SECONDS = 30
TENSORBOARD_LOG_GRAPH = True
# Also logs a readable full-model map with real stage shapes and parameter counts.
TENSORBOARD_LOG_ARCHITECTURE_IMAGE = True
TENSORBOARD_HISTOGRAM_INTERVAL = 5
TENSORBOARD_LOG_GRADIENT_HISTOGRAMS = True
TENSORBOARD_LOG_IMAGES = True
TENSORBOARD_LOG_PR_CURVES = True
TENSORBOARD_LOG_EMBEDDINGS = True
TENSORBOARD_PROFILE_BATCHES = 5
TENSORBOARD_MAX_PREDICTION_IMAGES = 16
TENSORBOARD_LOG_CLASS_METRICS = True
TENSORBOARD_LOG_SYSTEM_METRICS = True
TENSORBOARD_LOG_ACTIVATIONS = True
TENSORBOARD_ACTIVATION_INTERVAL = 5
TENSORBOARD_LOG_OPTIMIZER_STATE = True
TENSORBOARD_OPTIMIZER_INTERVAL = 5
TENSORBOARD_EMBEDDING_INTERVAL = 5
TENSORBOARD_MAX_EMBEDDING_SAMPLES = 512
TENSORBOARD_LOG_ROC_CURVES = True
TENSORBOARD_LOG_CALIBRATION = True
TENSORBOARD_LOG_GRADCAM = True
TENSORBOARD_MAX_GRADCAM_IMAGES = 4
# Full dual-backbone histograms are available but expensive; enable explicitly.
TENSORBOARD_LOG_FULL_MODEL_HISTOGRAMS = False

print(f"Runtime profile: {PLATFORM}")
print(f"Dataset source: {DATA_SOURCE}")

## 05. Resolve the dataset location

**This cell covers:** Reading an **Add Input** path, downloading a public dataset handle, extracting a single mounted archive when needed, and locating `train`, `validation`, and `test`.

Use one of these settings in Cell 04:

- `kaggle_input` for the normal **Add Input** workflow.
- `kaggle_handle` for a public dataset handle such as `owner/dataset-name`.
- `local_path` for data already available in the current session.

Private data should normally be attached with **Add Input**. Handle downloads can require internet access.


In [ ]:
# 05. Dataset path

import importlib.util
import subprocess
import tarfile
import zipfile


def ensure_package(import_name, package_name=None):
    """Install a small data-source helper only when it is missing."""
    if importlib.util.find_spec(import_name) is None:
        package_name = package_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


def has_required_splits(path):
    path = Path(path)
    return all((path / split).is_dir() for split in ("train", "validation", "test"))


def find_dataset_root(candidate):
    """Find the nearest folder containing all three required splits."""
    candidate = Path(candidate)
    if has_required_splits(candidate):
        return candidate
    if not candidate.exists():
        raise FileNotFoundError(f"Dataset path does not exist: {candidate}")
    for train_folder in candidate.rglob("train"):
        if train_folder.is_dir() and has_required_splits(train_folder.parent):
            return train_folder.parent
    raise FileNotFoundError(
        f"No folder containing train/validation/test was found under: {candidate}"
    )


def extract_archive(archive_path, output_folder):
    """Extract ZIP or TAR data even when a downloaded file has no suffix."""
    archive_path = Path(archive_path)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path.name} -> {output_folder}")

    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(output_folder)
    elif tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as archive:
            try:
                archive.extractall(output_folder, filter="data")
            except TypeError:
                archive.extractall(output_folder)
    else:
        shutil.unpack_archive(str(archive_path), str(output_folder))
    return output_folder


def prepare_dataset_path(candidate, work_folder):
    """Use a dataset folder directly or extract the one archive inside it."""
    candidate = Path(candidate)
    if candidate.is_file():
        return extract_archive(candidate, work_folder / "extracted")
    if has_required_splits(candidate):
        return candidate
    try:
        return find_dataset_root(candidate)
    except FileNotFoundError:
        archives = []
        for pattern in ("*.zip", "*.tar", "*.tar.gz", "*.tgz"):
            archives.extend(candidate.glob(pattern))
        if len(archives) == 1:
            return extract_archive(archives[0], work_folder / "extracted")
        raise



def resolve_data_dir():
    work_folder = CACHE_DIR / "datasets"
    for directory in (work_folder, TORCH_HOME_DIR):
        directory.mkdir(parents=True, exist_ok=True)
    os.environ["TORCH_HOME"] = str(TORCH_HOME_DIR)

    if DATA_SOURCE == "kaggle_input":
        candidate = Path(KAGGLE_INPUT_PATH)
        if not candidate.exists():
            available = sorted(str(path) for path in Path("/kaggle/input").glob("*"))
            raise FileNotFoundError(
                "The configured input path was not found. Use '+ Add Input', attach "
                f"the dataset, then update KAGGLE_INPUT_PATH. Available inputs: {available}"
            )

    elif DATA_SOURCE == "kaggle_handle":
        if not KAGGLE_DATASET_HANDLE:
            raise ValueError("Set KAGGLE_DATASET_HANDLE in Cell 04.")
        ensure_package("kagglehub")
        import kagglehub

        candidate = Path(kagglehub.dataset_download(KAGGLE_DATASET_HANDLE))

    elif DATA_SOURCE == "local_path":
        candidate = Path(LOCAL_DATASET_PATH)

    else:
        raise ValueError("DATA_SOURCE must be kaggle_input, kaggle_handle, or local_path.")

    prepared = prepare_dataset_path(candidate, work_folder)
    return find_dataset_root(prepared)


DATA_DIR = resolve_data_dir()
for directory in (
    RESULTS_DIR,
    BEST_MODELS_DIR,
    CHECKPOINTS_DIR,
    REPORTS_DIR,
    TENSORBOARD_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Dataset root: {DATA_DIR}")
print("Splits:", [item.name for item in DATA_DIR.iterdir() if item.is_dir()])
print(f"Results: {RESULTS_DIR}")
print(f"Best models: {BEST_MODELS_DIR}")
print(f"Checkpoints: {CHECKPOINTS_DIR}")
print(f"Reports: {REPORTS_DIR}")
print(f"TensorBoard logs: {TENSORBOARD_ROOT}")
print(f"Kaggle cache: {CACHE_DIR}")
print(f"Torch cache: {TORCH_HOME_DIR}")


## 06. Reproducibility and device setup

**This cell covers:** Random seeding, deterministic settings, CUDA device selection, and mixed-precision activation.

In [ ]:
# 06. Seed and device
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
CHANNELS_LAST_ENABLED = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print(
    f"Device: {DEVICE} | Mixed precision: {AMP_ENABLED} | "
    f"Channels last: {CHANNELS_LAST_ENABLED}"
)

## 07. Validate the dataset layout

**This cell covers:** Strict checks for the recommended `train`, `validation`, and `test` folder layout and matching class folders.

Expected structure:
```text
dataset/train/class_name/images...
dataset/validation/class_name/images...
dataset/test/class_name/images...
```

In [ ]:
# 07. Dataset layout
def validate_dataset_layout(data_dir):
    split_paths = {name: data_dir / name for name in ("train", "validation", "test")}
    missing = [name for name, path in split_paths.items() if not path.is_dir()]
    if missing:
        expected = "\n".join(str(data_dir / name / "<class_name>") for name in split_paths)
        raise FileNotFoundError(
            "UGVNet strongly recommends a tuned dataset with separate train, validation, "
            f"and test folders. Missing: {missing}\nExpected:\n{expected}"
        )
    class_sets = {}
    for name, path in split_paths.items():
        class_sets[name] = sorted(item.name for item in path.iterdir() if item.is_dir())
        if not class_sets[name]:
            raise ValueError(f"No class folders found in {path}")
    if not (class_sets["train"] == class_sets["validation"] == class_sets["test"]):
        raise ValueError("Class folders must match exactly across train, validation, and test.")
    return split_paths, class_sets["train"]

SPLIT_PATHS, DISCOVERED_CLASSES = validate_dataset_layout(DATA_DIR)
print(f"Validated {len(DISCOVERED_CLASSES)} classes: {DISCOVERED_CLASSES}")
print("Reminder: verify patient-level separation and remove duplicates before training.")

## 08. Audit the complete dataset

**This cell covers:** A full scan of every image before training. The audit decodes each file, calculates an exact SHA-256 fingerprint, reports split and class distributions, formats, color modes, dimensions, imbalance, corrupt images, within-split duplicates, and train/validation/test leakage.

The complete report is saved to `RESULTS_DIR / "dataset_audit.json"` and displayed before DataLoaders or the model are created.

- `AUDIT_POLICY = "strict"` stops before training when corrupt files, cross-split duplicates, or other critical issues are found.
- `AUDIT_POLICY = "warn"` displays the problems but allows training.
- `AUDIT_POLICY = "off"` skips the scan and is not recommended.

The scanner reads each image once, then reuses those bytes for exact SHA-256 duplicate detection and full image decoding. This keeps corruption and leakage checks exact while reducing storage I/O.

In [ ]:
# 08. Dataset audit
import hashlib
import itertools
import statistics
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO


def _inspect_image(item):
    path, split, class_name = item
    relative = path.relative_to(DATA_DIR).as_posix()
    try:
        payload = path.read_bytes()
        digest = hashlib.sha256(payload).hexdigest()
        with Image.open(BytesIO(payload)) as image:
            width, height = image.size
            image_format = image.format or path.suffix.lstrip(".").upper()
            mode = image.mode
            image.load()
        return {
            "path": relative,
            "split": split,
            "class_name": class_name,
            "width": width,
            "height": height,
            "format": image_format,
            "mode": mode,
            "sha256": digest,
            "error": None,
        }
    except Exception as error:  # noqa: BLE001
        return {
            "path": relative,
            "split": split,
            "class_name": class_name,
            "width": None,
            "height": None,
            "format": None,
            "mode": None,
            "sha256": None,
            "error": f"{type(error).__name__}: {error}",
        }


def audit_dataset():
    supported = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".gif"}
    work = []
    ignored = []
    for split, split_path in SPLIT_PATHS.items():
        for class_folder in sorted(path for path in split_path.iterdir() if path.is_dir()):
            for path in sorted(class_folder.rglob("*")):
                if not path.is_file():
                    continue
                if path.suffix.lower() in supported:
                    work.append((path, split, class_folder.name))
                else:
                    ignored.append(path.relative_to(DATA_DIR).as_posix())
    if not work:
        raise RuntimeError("The audit found no supported image files.")

    print(f"Scanning and decoding all {len(work):,} images with {AUDIT_WORKERS} workers...")
    records = []
    started = time.time()
    with ThreadPoolExecutor(max_workers=AUDIT_WORKERS) as executor:
        for index, record in enumerate(executor.map(_inspect_image, work), start=1):
            records.append(record)
            if index % 1000 == 0 or index == len(work):
                print(f"  scanned {index:,}/{len(work):,}")

    valid = [record for record in records if record["error"] is None]
    corrupt = [record for record in records if record["error"] is not None]
    hashes = defaultdict(list)
    for record in valid:
        hashes[record["sha256"]].append(record)
    duplicates = [group for group in hashes.values() if len(group) > 1]
    cross_split = [
        group for group in duplicates
        if len({record["split"] for record in group}) > 1
    ]
    within_split = [
        group for group in duplicates
        if len({record["split"] for record in group}) == 1
    ]

    class_counts = {
        split: dict(Counter(record["class_name"] for record in records if record["split"] == split))
        for split in ("train", "validation", "test")
    }
    train_counts = [count for count in class_counts["train"].values() if count > 0]
    imbalance = max(train_counts) / min(train_counts) if train_counts else float("inf")
    widths = [record["width"] for record in valid]
    heights = [record["height"] for record in valid]
    formats = dict(Counter(record["format"] for record in valid).most_common())
    modes = dict(Counter(record["mode"] for record in valid).most_common())
    critical_issues = len(corrupt) + len(cross_split)

    report = {
        "dataset_root": str(DATA_DIR),
        "duration_seconds": round(time.time() - started, 3),
        "summary": {
            "candidate_images": len(records),
            "valid_images": len(valid),
            "corrupt_images": len(corrupt),
            "ignored_files": len(ignored),
            "exact_duplicate_groups": len(duplicates),
            "within_split_duplicate_groups": len(within_split),
            "cross_split_duplicate_groups": len(cross_split),
            "critical_issue_count": critical_issues,
        },
        "class_counts": class_counts,
        "training_imbalance_ratio": round(imbalance, 4),
        "image_properties": {
            "width": {
                "min": min(widths) if widths else None,
                "median": statistics.median(widths) if widths else None,
                "max": max(widths) if widths else None,
            },
            "height": {
                "min": min(heights) if heights else None,
                "median": statistics.median(heights) if heights else None,
                "max": max(heights) if heights else None,
            },
            "formats": formats,
            "color_modes": modes,
        },
        "issues": {
            "corrupt_images": corrupt,
            "cross_split_duplicates": [[item["path"] for item in group] for group in cross_split],
            "within_split_duplicates": [[item["path"] for item in group] for group in within_split],
            "ignored_file_examples": ignored[:100],
        },
    }
    report_path = RESULTS_DIR / "dataset_audit.json"
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

    # ── Formatted console report ──
    _hl = "─" * 72
    print(f"\n┌{_hl}┐")
    print(f"│{'UGVNet PRE-TRAINING DATASET AUDIT':^72s}│")
    print(f"├{_hl}┤")
    print(
        f"│  {'Images':16s} │ {len(records):>8,} total  "
        f"│ {len(valid):>8,} valid   "
        f"│ {len(corrupt):>5,} corrupt │"
    )
    print(f"│  {'Ignored files':16s} │ {len(ignored):>8,}        │             │             │")
    _fmt_str = ", ".join(f"{k} ({v:,})" for k, v in formats.items())
    print(f"│  {'Formats':16s} │ {_fmt_str:>50s} │")
    _mode_str = ", ".join(f"{k} ({v:,})" for k, v in modes.items())
    print(f"│  {'Color modes':16s} │ {_mode_str:>50s} │")
    _w = report['image_properties']['width']
    _h = report['image_properties']['height']
    _w_str = f"min {_w['min']}  ·  median {_w['median']}  ·  max {_w['max']}"
    _h_str = f"min {_h['min']}  ·  median {_h['median']}  ·  max {_h['max']}"
    print(f"│  {'Width  (px)':16s} │ {_w_str:<50s} │")
    print(f"│  {'Height (px)':16s} │ {_h_str:<50s} │")
    print(f"│  {'Imbalance ratio':16s} │ {imbalance:>49.2f}x │")
    _dup_str = f"{len(duplicates):,} total   ·   {len(cross_split):,} crossing splits"
    print(f"│  {'Duplicate groups':16s} │ {_dup_str:<50s} │")
    print(f"├{_hl}┤")
    print(f"│  {'Class':18s} {'Train':>10s} {'Validation':>12s} {'Test':>10s} {'Total':>10s}      │")
    print(f"│  {'─' * 62:64s}│")
    for class_name in DISCOVERED_CLASSES:
        _tr = class_counts["train"].get(class_name, 0)
        _va = class_counts["validation"].get(class_name, 0)
        _te = class_counts["test"].get(class_name, 0)
        print(f"│  {class_name:18s} {_tr:>10,} {_va:>12,} {_te:>10,} {_tr+_va+_te:>10,}      │")
    if corrupt:
        print(f"├{_hl}┤")
        print(f"│  Corrupt-image examples:{'':>48s}│")
        for item in corrupt[:10]:
            _msg = f"  · {item['path']}: {item['error']}"
            print(f"│{_msg:<72s}│")
    if cross_split:
        print(f"├{_hl}┤")
        print(f"│  Cross-split duplicate examples:{'':>40s}│")
        for group in cross_split[:10]:
            _msg = "  · " + " ↔ ".join(item["path"] for item in group)
            print(f"│{_msg:<72s}│")
    print(f"├{_hl}┤")
    print(f"│  Report: {str(report_path):<62s}│")
    _status = "✓ PASS" if critical_issues == 0 else "⚠ ATTENTION REQUIRED"
    print(f"│  Status: {_status:<62s}│")
    print(f"└{_hl}┘")

    # ── Styled DataFrames and CSV export ──
    audit_summary_table = pd.DataFrame([
        {"Measure": key.replace("_", " ").title(), "Value": value}
        for key, value in report["summary"].items()
    ])
    audit_class_table = pd.DataFrame([
        {
            "Class": class_name,
            "Train": class_counts["train"].get(class_name, 0),
            "Validation": class_counts["validation"].get(class_name, 0),
            "Test": class_counts["test"].get(class_name, 0),
        }
        for class_name in DISCOVERED_CLASSES
    ])
    audit_class_table["Total"] = audit_class_table[
        ["Train", "Validation", "Test"]
    ].sum(axis=1)
    audit_summary_table.to_csv(
        RESULTS_DIR / "dataset_audit_summary.csv", index=False,
    )
    audit_class_table.to_csv(
        RESULTS_DIR / "dataset_class_distribution.csv", index=False,
    )
    show_dataframe(audit_summary_table, "Pre-training dataset audit")
    show_dataframe(audit_class_table, "Complete dataset class distribution")

    # ── Enhanced bar chart with value labels ──
    split_colors = {"train": "#93c5fd", "validation": "#86efac", "test": "#fca5a5"}
    split_edge   = {"train": "#3b82f6", "validation": "#22c55e", "test": "#ef4444"}
    x = np.arange(len(DISCOVERED_CLASSES))
    bar_width = 0.25
    figure, axis = plt.subplots(
        figsize=(max(10, len(DISCOVERED_CLASSES) * 1.4), 5.5),
        facecolor="#fafbff",
    )
    axis.set_facecolor("#fafbff")
    bars_by_split = {}
    for offset, split in enumerate(("train", "validation", "test")):
        values = [class_counts[split].get(name, 0) for name in DISCOVERED_CLASSES]
        split_total = sum(values)
        bars = axis.bar(
            x + (offset - 1) * bar_width,
            values,
            bar_width,
            label=f"{split} ({split_total:,})",
            color=split_colors[split],
            edgecolor=split_edge[split],
            linewidth=0.8,
        )
        bars_by_split[split] = bars
        # Add value labels on each bar
        for bar_rect in bars:
            height = bar_rect.get_height()
            if height > 0:
                axis.text(
                    bar_rect.get_x() + bar_rect.get_width() / 2,
                    height + max(max(class_counts["train"].values()) * 0.01, 5),
                    f"{int(height):,}",
                    ha="center",
                    va="bottom",
                    fontsize=7.5,
                    fontweight="600",
                    color="#334155",
                    fontfamily="Inter",
                )
    axis.set_xticks(x)
    axis.set_xticklabels(DISCOVERED_CLASSES, rotation=40, ha="right", fontsize=11, fontfamily="Inter")
    axis.set_ylabel("Number of Images", fontsize=12, fontfamily="Inter", fontweight="500")
    axis.set_title(
        "Complete Dataset Class Distribution",
        fontsize=15,
        fontfamily="Inter",
        fontweight="700",
        pad=14,
    )
    axis.legend(
        frameon=True,
        fancybox=True,
        shadow=False,
        framealpha=0.85,
        edgecolor="#cbd5e1",
        fontsize=10,
        prop={"family": "Inter", "weight": "500"},
    )
    axis.grid(axis="y", linestyle="--", alpha=0.4, color="#94a3b8")
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_color("#cbd5e1")
    axis.spines["bottom"].set_color("#cbd5e1")
    axis.tick_params(axis="y", labelsize=10)
    figure.tight_layout()
    figure.savefig(
        RESULTS_DIR / "dataset_class_distribution.png",
        dpi=160,
        bbox_inches="tight",
        facecolor="#fafbff",
    )
    plt.show()
    return report


if AUDIT_POLICY == "off":
    DATASET_AUDIT = None
    print("WARNING: complete dataset audit is disabled.")
else:
    DATASET_AUDIT = audit_dataset()
    critical = DATASET_AUDIT["summary"]["critical_issue_count"]
    if critical and AUDIT_POLICY == "strict":
        raise RuntimeError(
            f"Dataset audit found {critical} critical issue(s). Correct the dataset "
            "before training, or deliberately change AUDIT_POLICY to 'warn'."
        )
    if critical:
        print(f"WARNING: continuing with {critical} critical issue(s).")

## 09. Transforms and datasets

**This cell covers:** Automatic small/large-dataset detection, the faster large-data profile, standard augmentation, stronger class-aware augmentation for clearly underrepresented classes, deterministic validation/test preprocessing, and ImageFolder creation.

Minority augmentation is selected from the training counts only. It changes augmentation diversity, never the validation or test distribution.


In [ ]:
# 09. Transforms and datasets
provisional_train = datasets.ImageFolder(SPLIT_PATHS["train"])
if TRAINING_MODE == "auto":
    RESOLVED_MODE = (
        "small" if len(provisional_train) < SMALL_DATASET_THRESHOLD else "large"
    )
elif TRAINING_MODE in {"small", "large"}:
    RESOLVED_MODE = TRAINING_MODE
else:
    raise ValueError("TRAINING_MODE must be auto, small, or large.")

if RESOLVED_MODE == "large":
    IMAGE_SIZE = LARGE_DATASET_IMAGE_SIZE
    FUSION_CHANNELS = LARGE_DATASET_FUSION_CHANNELS
    FUSION_DEPTH = LARGE_DATASET_FUSION_DEPTH
else:
    IMAGE_SIZE = SMALL_DATASET_IMAGE_SIZE
    FUSION_CHANNELS = SMALL_DATASET_FUSION_CHANNELS
    FUSION_DEPTH = SMALL_DATASET_FUSION_DEPTH

normalize = transforms.Normalize(
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
)
crop_scale = (0.60, 1.0) if RESOLVED_MODE == "small" else (0.75, 1.0)
augmentation_magnitude = 7 if RESOLVED_MODE == "small" else 5

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=crop_scale),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.RandAugment(num_ops=2, magnitude=augmentation_magnitude),
    transforms.ToTensor(),
    normalize,
    transforms.RandomErasing(
        p=0.20 if RESOLVED_MODE == "small" else 0.10,
        scale=(0.02, 0.12),
    ),
])
minority_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.50, 1.0) if RESOLVED_MODE == "small" else (0.65, 1.0),
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.RandAugment(
        num_ops=2,
        magnitude=min(10, augmentation_magnitude + 2),
    ),
    transforms.ToTensor(),
    normalize,
    transforms.RandomErasing(
        p=0.30 if RESOLVED_MODE == "small" else 0.18,
        scale=(0.02, 0.15),
    ),
])
evaluation_transform = transforms.Compose([
    transforms.Resize(round(IMAGE_SIZE * 1.14)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    normalize,
])

if MINORITY_AUGMENTATION not in {"auto", "on", "off"}:
    raise ValueError("MINORITY_AUGMENTATION must be auto, on, or off.")
provisional_counts = Counter(provisional_train.targets)
maximum_train_count = max(provisional_counts.values())
minimum_train_count = min(provisional_counts.values())
provisional_imbalance_ratio = maximum_train_count / minimum_train_count
minority_augmentation_enabled = MINORITY_AUGMENTATION == "on" or (
    MINORITY_AUGMENTATION == "auto" and provisional_imbalance_ratio >= 1.5
)
minority_cutoff = maximum_train_count * MINORITY_AUGMENTATION_MAX_FRACTION
MINORITY_CLASS_INDICES = {
    class_index
    for class_index, class_count in provisional_counts.items()
    if minority_augmentation_enabled and class_count <= minority_cutoff
}
if (
    MINORITY_AUGMENTATION == "on"
    and not MINORITY_CLASS_INDICES
    and maximum_train_count > minimum_train_count
):
    MINORITY_CLASS_INDICES = {
        class_index
        for class_index, class_count in provisional_counts.items()
        if class_count == minimum_train_count
    }


class ClassAwareTransformDataset(torch.utils.data.Dataset):
    """Use stronger augmentation only for clearly underrepresented classes."""

    def __init__(self, base_dataset, standard_transform, minority_transform, minority_indices):
        self.base_dataset = base_dataset
        self.standard_transform = standard_transform
        self.minority_transform = minority_transform
        self.minority_indices = set(minority_indices)
        self.classes = base_dataset.classes
        self.class_to_idx = base_dataset.class_to_idx
        self.samples = base_dataset.samples
        self.imgs = base_dataset.imgs
        self.targets = base_dataset.targets

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        image, target = self.base_dataset[index]
        transform = (
            self.minority_transform
            if target in self.minority_indices
            else self.standard_transform
        )
        return transform(image), target


DATASETS = {
    "train": ClassAwareTransformDataset(
        provisional_train,
        train_transform,
        minority_train_transform,
        MINORITY_CLASS_INDICES,
    ),
    "validation": datasets.ImageFolder(
        SPLIT_PATHS["validation"], evaluation_transform
    ),
    "test": datasets.ImageFolder(SPLIT_PATHS["test"], evaluation_transform),
}
CLASSES = DATASETS["train"].classes
NUM_CLASSES = len(CLASSES)
for split in ("validation", "test"):
    if DATASETS[split].classes != CLASSES:
        raise ValueError(f"{split} classes do not match training classes.")

print(
    f"Training mode: {RESOLVED_MODE} | Input: {IMAGE_SIZE}x{IMAGE_SIZE} | "
    f"Fusion: {FUSION_CHANNELS} channels x {FUSION_DEPTH} block(s)"
)
for split, dataset in DATASETS.items():
    print(f"{split:10s}: {len(dataset):7,d} images")
print(
    "Minority-class augmentation: "
    + (
        ", ".join(CLASSES[index] for index in sorted(MINORITY_CLASS_INDICES))
        if MINORITY_CLASS_INDICES
        else "not required"
    )
)


## 10. Class balance and data loaders

**This cell covers:** Class-count reporting, automatic imbalance detection, effective-number sampling, class-weight reporting, pinned-memory prefetching, persistent workers, and larger validation/test batches.

Effective-number weights are gentler than raw inverse-frequency weights and reduce repeated sampling of the smallest classes. The loaders retain the existing fast prefetch and evaluation-batch settings.


In [ ]:
# 10. Data loaders
import pandas as pd
counts = Counter(DATASETS["train"].targets)

distribution_df = pd.DataFrame([
    {"Index": index, "Class": class_name, "Count": counts[index]}
    for index, class_name in enumerate(CLASSES)
])
show_dataframe(distribution_df, "Training class distribution")

imbalance_ratio = max(counts.values()) / min(counts.values())
use_balanced_sampler = CLASS_BALANCE == "on" or (
    CLASS_BALANCE == "auto" and imbalance_ratio >= 1.5
)
sampler = None
resolved_sampler_beta = None
sampler_class_weights = torch.ones(NUM_CLASSES, dtype=torch.double)
if use_balanced_sampler:
    if BALANCED_SAMPLER_STRATEGY == "effective_number":
        if EFFECTIVE_NUMBER_BETA == "auto":
            resolved_sampler_beta = max(
                0.90,
                min(0.9999, 1.0 - 1.0 / len(DATASETS["train"])),
            )
        else:
            resolved_sampler_beta = float(EFFECTIVE_NUMBER_BETA)
        if not 0.0 <= resolved_sampler_beta < 1.0:
            raise ValueError("EFFECTIVE_NUMBER_BETA must be in [0, 1).")
        class_count_tensor = torch.tensor(
            [counts[index] for index in range(NUM_CLASSES)],
            dtype=torch.double,
        )
        effective_numbers = 1.0 - resolved_sampler_beta**class_count_tensor
        sampler_class_weights = (1.0 - resolved_sampler_beta) / effective_numbers
        sampler_class_weights /= sampler_class_weights.mean()
    elif BALANCED_SAMPLER_STRATEGY == "inverse_frequency":
        sampler_class_weights = torch.tensor(
            [1.0 / counts[index] for index in range(NUM_CLASSES)],
            dtype=torch.double,
        )
        sampler_class_weights /= sampler_class_weights.mean()
    else:
        raise ValueError(
            "BALANCED_SAMPLER_STRATEGY must be effective_number or inverse_frequency."
        )
    sample_weights = sampler_class_weights[
        torch.tensor(DATASETS["train"].targets, dtype=torch.long)
    ]
    sampler = WeightedRandomSampler(
        sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

common_loader_options = {
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
    "persistent_workers": NUM_WORKERS > 0,
}
if NUM_WORKERS > 0:
    common_loader_options["prefetch_factor"] = PREFETCH_FACTOR

LOADERS = {
    "train": DataLoader(
        DATASETS["train"],
        batch_size=BATCH_SIZE,
        shuffle=sampler is None,
        sampler=sampler,
        **common_loader_options,
    ),
    "validation": DataLoader(
        DATASETS["validation"],
        batch_size=BATCH_SIZE * EVAL_BATCH_MULTIPLIER,
        shuffle=False,
        **common_loader_options,
    ),
    "test": DataLoader(
        DATASETS["test"],
        batch_size=BATCH_SIZE * EVAL_BATCH_MULTIPLIER,
        shuffle=False,
        **common_loader_options,
    ),
}
loader_info = pd.DataFrame([
    {"Metric": "Imbalance ratio", "Value": f"{imbalance_ratio:.2f}x"},
    {"Metric": "Balanced sampling", "Value": str(use_balanced_sampler)},
    {"Metric": "Strategy", "Value": BALANCED_SAMPLER_STRATEGY if use_balanced_sampler else 'none'},
    {"Metric": "Train batch", "Value": str(BATCH_SIZE)},
    {"Metric": "Evaluation batch", "Value": str(BATCH_SIZE * EVAL_BATCH_MULTIPLIER)},
    {"Metric": "Workers", "Value": str(NUM_WORKERS)},
    {"Metric": "Prefetch", "Value": str(PREFETCH_FACTOR if NUM_WORKERS > 0 else 0)},
])
show_dataframe(loader_info, "Data Loader Configuration")

if use_balanced_sampler:
    beta_text = (
        f" (beta: {resolved_sampler_beta:.6f})"
        if resolved_sampler_beta is not None
        else ""
    )
    weights_df = pd.DataFrame([
        {"Class": class_name, "Weight": weight}
        for class_name, weight in zip(CLASSES, sampler_class_weights.tolist())
    ])


## 11. Fusion and attention layers

**This cell covers:** Channel-wise 2D normalization, global spatial self-attention, adaptive two-backbone gating, and the global refinement block.

In [ ]:
# 11. Fusion layers
class LayerNorm2d(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.norm = nn.LayerNorm(channels, eps=eps)

    def forward(self, x):
        return self.norm(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2).contiguous()


class GlobalSelfAttention(nn.Module):
    def __init__(self, channels, num_heads, dropout=0.0):
        super().__init__()
        if channels % num_heads != 0:
            raise ValueError("channels must be divisible by num_heads")
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.attention_dropout = nn.Dropout(dropout)
        self.projection = nn.Conv2d(channels, channels, 1)
        self.projection_dropout = nn.Dropout(dropout)
        self.capture_tensorboard_stats = False
        self.last_attention_entropy = None

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = height * width
        qkv = self.qkv(x).reshape(
            batch, 3, self.num_heads, self.head_dim, tokens
        )
        query, key, value = qkv.unbind(dim=1)
        attention = ((query.transpose(-2, -1) * self.scale) @ key).softmax(dim=-1)
        if self.capture_tensorboard_stats:
            stable_attention = attention.float().clamp_min(1e-8)
            self.last_attention_entropy = (
                -(stable_attention * stable_attention.log()).sum(dim=-1)
                / math.log(max(tokens, 2))
            ).mean().detach()
        else:
            self.last_attention_entropy = None
        attention = self.attention_dropout(attention)
        output = value @ attention.transpose(-2, -1)
        output = output.reshape(batch, channels, height, width)
        return self.projection_dropout(self.projection(output))


class AdaptiveBackboneFusion(nn.Module):
    def __init__(self, fusion_channels):
        super().__init__()
        self.efficientnet_projection = nn.Sequential(
            nn.Conv2d(1280, fusion_channels, 1),
            nn.GroupNorm(1, fusion_channels),
            nn.GELU(),
        )
        self.convnext_projection = nn.Sequential(
            nn.Conv2d(768, fusion_channels, 1),
            nn.GroupNorm(1, fusion_channels),
            nn.GELU(),
        )
        self.gate = nn.Sequential(
            nn.Conv2d(fusion_channels * 2, fusion_channels, 1),
            nn.GELU(),
            nn.Conv2d(fusion_channels, 2, 1),
        )
        self.capture_tensorboard_stats = False
        self.last_tensorboard_stats = {}

    def forward(self, efficientnet_features, convnext_features):
        efficientnet_features = self.efficientnet_projection(efficientnet_features)
        convnext_features = self.convnext_projection(convnext_features)
        if efficientnet_features.shape[-2:] != convnext_features.shape[-2:]:
            efficientnet_features = F.interpolate(
                efficientnet_features,
                size=convnext_features.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        weights = self.gate(
            torch.cat([efficientnet_features, convnext_features], dim=1)
        ).softmax(dim=1)
        if self.capture_tensorboard_stats:
            with torch.no_grad():
                stable_weights = weights.float().clamp_min(1e-8)
                efficientnet_flat = efficientnet_features.float().flatten(1)
                convnext_flat = convnext_features.float().flatten(1)
                self.last_tensorboard_stats = {
                    "efficientnet_gate_weight": weights[:, 0].float().mean().detach(),
                    "convnext_gate_weight": weights[:, 1].float().mean().detach(),
                    "gate_entropy": (
                        -(stable_weights * stable_weights.log()).sum(dim=1)
                        / math.log(2.0)
                    ).mean().detach(),
                    "gate_dominance": (
                        weights[:, 0].float() - weights[:, 1].float()
                    ).abs().mean().detach(),
                    "efficientnet_selection_rate": (
                        weights[:, 0] > weights[:, 1]
                    ).float().mean().detach(),
                    "efficientnet_feature_norm": efficientnet_flat.norm(
                        dim=1
                    ).mean().detach(),
                    "convnext_feature_norm": convnext_flat.norm(
                        dim=1
                    ).mean().detach(),
                    "branch_feature_cosine_similarity": F.cosine_similarity(
                        efficientnet_flat,
                        convnext_flat,
                        dim=1,
                    ).mean().detach(),
                }
        else:
            self.last_tensorboard_stats = {}
        fused = weights[:, :1] * efficientnet_features + weights[:, 1:] * convnext_features
        return fused, weights


class GlobalFusionBlock(nn.Module):
    def __init__(self, channels, num_heads, dropout):
        super().__init__()
        hidden = channels * 4
        self.position = nn.Conv2d(channels, channels, 3, padding=1, groups=channels)
        self.attention_norm = LayerNorm2d(channels)
        self.attention = GlobalSelfAttention(channels, num_heads, dropout)
        self.mlp_norm = LayerNorm2d(channels)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv2d(hidden, channels, 1),
            nn.Dropout(dropout),
        )
        self.attention_scale = nn.Parameter(torch.full((channels, 1, 1), 1e-5))
        self.mlp_scale = nn.Parameter(torch.full((channels, 1, 1), 1e-5))

    def forward(self, x):
        x = x + self.position(x)
        x = x + self.attention_scale * self.attention(self.attention_norm(x))
        x = x + self.mlp_scale * self.mlp(self.mlp_norm(x))
        return x

## 12. UGVNet model

**This cell covers:** The complete EfficientNetV2-S + ConvNeXt-Tiny model, pretrained feature extraction, adaptive fusion, global refinement, classifier, and backbone freezing controls.

In [ ]:
# 12. Model
class UGVNetHybrid(nn.Module):
    def __init__(
        self,
        num_classes,
        pretrained=True,
        fusion_channels=384,
        attention_heads=8,
        fusion_depth=2,
        dropout=0.2,
    ):
        super().__init__()
        efficientnet_weights = EfficientNet_V2_S_Weights.DEFAULT if pretrained else None
        convnext_weights = ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        efficientnet = efficientnet_v2_s(weights=efficientnet_weights)
        convnext = convnext_tiny(weights=convnext_weights)
        self.efficientnet_features = efficientnet.features
        self.convnext_features = convnext.features
        self.fusion = AdaptiveBackboneFusion(fusion_channels)
        self.global_fusion = nn.Sequential(*[
            GlobalFusionBlock(fusion_channels, attention_heads, dropout)
            for _ in range(fusion_depth)
        ])
        self.final_norm = LayerNorm2d(fusion_channels)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(fusion_channels, num_classes)

    def extract_backbone_features(self, x):
        return self.efficientnet_features(x), self.convnext_features(x)

    def forward_features(self, x, return_fusion_weights=False):
        efficientnet_features, convnext_features = self.extract_backbone_features(x)
        fused, weights = self.fusion(efficientnet_features, convnext_features)
        fused = self.final_norm(self.global_fusion(fused))
        return (fused, weights) if return_fusion_weights else fused

    def forward_head(self, features):
        """Used for Grad-CAM to map fused spatial features to logits."""
        embedding = self.pool(features).flatten(1)
        return self.head(self.dropout(embedding))

    def forward(self, x):
        features = self.forward_features(x)
        return self.forward_head(features)

    def set_backbones_trainable(self, trainable):
        for backbone in (self.efficientnet_features, self.convnext_features):
            for parameter in backbone.parameters():
                parameter.requires_grad = trainable

    def train(self, mode=True):
        super().train(mode)
        if mode:
            if not any(p.requires_grad for p in self.efficientnet_features.parameters()):
                self.efficientnet_features.eval()
            if not any(p.requires_grad for p in self.convnext_features.parameters()):
                self.convnext_features.eval()
        return self

    def backbone_parameters(self):
        yield from self.efficientnet_features.parameters()
        yield from self.convnext_features.parameters()

    def new_parameters(self):
        yield from self.fusion.parameters()
        yield from self.global_fusion.parameters()
        yield from self.final_norm.parameters()
        yield from self.head.parameters()

## 13. Training setup

**This cell covers:** Model creation, profile-specific backbone warm-up, channels-last CUDA layout, discriminative learning rates, automatic cross-entropy/focal-loss selection, cosine scheduling, mixed precision, and validated full-state checkpoint resume.

Focal loss is unweighted and activates automatically only for severe imbalance, so the sampler and loss do not apply duplicate class weighting. Set `RESUME_CHECKPOINT_PATH` in Cell 04 to restore a trusted UGVNet checkpoint before continuing.

TensorBoard is initialized once per run. This cell writes the complete traced operation graph with named expandable sections plus a readable architecture image containing both backbones, fusion and classifier stages, real output shapes, and parameter counts.


In [ ]:
# 13. Optimizer and loss
model = UGVNetHybrid(
    num_classes=NUM_CLASSES,
    pretrained=PRETRAINED,
    fusion_channels=FUSION_CHANNELS,
    attention_heads=ATTENTION_HEADS,
    fusion_depth=FUSION_DEPTH,
    dropout=DROPOUT,
).to(DEVICE)

if CHANNELS_LAST_ENABLED:
    model = model.to(memory_format=torch.channels_last)

freeze_epochs = (
    SMALL_DATASET_FREEZE_EPOCHS
    if RESOLVED_MODE == "small"
    else LARGE_DATASET_FREEZE_EPOCHS
)
if freeze_epochs > 0:
    model.set_backbones_trainable(False)

optimizer_groups = [
    {"params": list(model.backbone_parameters()), "lr": BACKBONE_LR},
    {"params": list(model.new_parameters()), "lr": FUSION_LR},
]
FUSED_ADAMW_ENABLED = False
if USE_FUSED_ADAMW and DEVICE.type == "cuda":
    try:
        optimizer = AdamW(
            optimizer_groups,
            weight_decay=WEIGHT_DECAY,
            fused=True,
        )
        FUSED_ADAMW_ENABLED = True
    except (TypeError, RuntimeError):
        optimizer = AdamW(optimizer_groups, weight_decay=WEIGHT_DECAY)
else:
    optimizer = AdamW(optimizer_groups, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)


class FocalCrossEntropyLoss(nn.Module):
    """Focal cross-entropy without a second layer of class weighting."""

    def __init__(self, gamma=1.5, label_smoothing=0.0):
        super().__init__()
        self.gamma = float(gamma)
        self.label_smoothing = float(label_smoothing)

    def forward(self, logits, targets):
        cross_entropy = F.cross_entropy(
            logits,
            targets,
            reduction="none",
            label_smoothing=self.label_smoothing,
        )
        true_class_probability = torch.softmax(logits, dim=1).gather(
            1,
            targets.unsqueeze(1),
        ).squeeze(1)
        focal_weight = (1.0 - true_class_probability).pow(self.gamma)
        return (focal_weight * cross_entropy).mean()


if LOSS_FUNCTION not in {"auto", "cross_entropy", "focal"}:
    raise ValueError("LOSS_FUNCTION must be auto, cross_entropy, or focal.")
use_focal_loss = LOSS_FUNCTION == "focal" or (
    LOSS_FUNCTION == "auto"
    and imbalance_ratio >= FOCAL_LOSS_IMBALANCE_THRESHOLD
)
RESOLVED_LOSS_FUNCTION = "focal" if use_focal_loss else "cross_entropy"


def build_criterion(label_smoothing, focal_gamma=None):
    if use_focal_loss:
        resolved_gamma = FOCAL_GAMMA if focal_gamma is None else focal_gamma
        return FocalCrossEntropyLoss(
            gamma=resolved_gamma,
            label_smoothing=label_smoothing,
        )
    return nn.CrossEntropyLoss(label_smoothing=label_smoothing)


current_label_smoothing = LABEL_SMOOTHING
current_dropout = DROPOUT
current_focal_gamma = FOCAL_GAMMA
criterion = build_criterion(current_label_smoothing, current_focal_gamma)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


def set_model_dropout(module, probability):
    for layer in module.modules():
        if isinstance(layer, nn.Dropout):
            layer.p = probability


resume_checkpoint = None
resume_checkpoint_value = str(RESUME_CHECKPOINT_PATH).strip()
if resume_checkpoint_value:
    resume_path = Path(resume_checkpoint_value).expanduser()
    if not resume_path.is_file():
        raise FileNotFoundError(f"Resume checkpoint was not found: {resume_path}")
    resume_checkpoint = torch.load(resume_path, map_location=DEVICE, weights_only=False)
    required_resume_keys = {"model", "optimizer", "scheduler", "epoch"}
    missing_resume_keys = required_resume_keys.difference(resume_checkpoint)
    if missing_resume_keys:
        raise ValueError(
            "Resume checkpoint is missing: "
            + ", ".join(sorted(missing_resume_keys))
        )
    checkpoint_classes = resume_checkpoint.get("classes")
    if checkpoint_classes is not None and list(checkpoint_classes) != list(CLASSES):
        raise ValueError("Resume checkpoint classes do not match this dataset.")
    # Legacy UGVNet checkpoints predate loss metadata and used cross-entropy.
    checkpoint_loss = resume_checkpoint.get("loss_function", "cross_entropy")
    if checkpoint_loss and checkpoint_loss != RESOLVED_LOSS_FUNCTION:
        raise ValueError(
            f"Checkpoint loss is {checkpoint_loss}, but this run resolved to "
            f"{RESOLVED_LOSS_FUNCTION}. Set LOSS_FUNCTION to match the checkpoint."
        )
    checkpoint_config = resume_checkpoint.get("model_config", {})
    expected_model_config = {
        "num_classes": NUM_CLASSES,
        "image_size": IMAGE_SIZE,
        "fusion_channels": FUSION_CHANNELS,
        "attention_heads": ATTENTION_HEADS,
        "fusion_depth": FUSION_DEPTH,
    }
    mismatches = {
        key: (checkpoint_config.get(key), expected_value)
        for key, expected_value in expected_model_config.items()
        if key in checkpoint_config and checkpoint_config[key] != expected_value
    }
    if mismatches:
        raise ValueError(f"Resume checkpoint model configuration mismatch: {mismatches}")

    model.load_state_dict(resume_checkpoint["model"])
    optimizer.load_state_dict(resume_checkpoint["optimizer"])
    scheduler.load_state_dict(resume_checkpoint["scheduler"])
    if "scaler" in resume_checkpoint:
        scaler.load_state_dict(resume_checkpoint["scaler"])
    model.set_backbones_trainable(
        resume_checkpoint.get("backbones_trainable", True)
    )
    current_dropout = resume_checkpoint.get("runtime_dropout", DROPOUT)
    current_label_smoothing = resume_checkpoint.get(
        "runtime_label_smoothing",
        LABEL_SMOOTHING,
    )
    current_focal_gamma = resume_checkpoint.get(
        "runtime_focal_gamma",
        FOCAL_GAMMA,
    )
    set_model_dropout(model, current_dropout)
    criterion = build_criterion(
        current_label_smoothing,
        current_focal_gamma,
    )

    rng_state = resume_checkpoint.get("rng_state", {})
    if "python" in rng_state:
        random.setstate(rng_state["python"])
    if "numpy" in rng_state:
        numpy_state = list(rng_state["numpy"])
        numpy_state[1] = np.asarray(numpy_state[1], dtype=np.uint32)
        np.random.set_state(tuple(numpy_state))
    if "torch" in rng_state:
        torch.set_rng_state(rng_state["torch"].cpu())
    if DEVICE.type == "cuda" and rng_state.get("cuda"):
        torch.cuda.set_rng_state_all([state.cpu() for state in rng_state["cuda"]])
    UGVNET_RESUME_STATE = resume_checkpoint
    print(
        f"Resume checkpoint loaded: {resume_path} | "
        f"epoch {resume_checkpoint['epoch']}"
    )


total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_parameters:,}")
print(f"Initially trainable:  {trainable_parameters:,}")
print(f"Frozen warm-up:       {freeze_epochs} epoch(s)")
print(
    f"Effective profile:    {IMAGE_SIZE}px, {FUSION_CHANNELS} fusion channels, "
    f"{FUSION_DEPTH} fusion block(s)"
)
print(f"Fused AdamW:          {FUSED_ADAMW_ENABLED}")
print(f"Loss function:        {RESOLVED_LOSS_FUNCTION}")
print(
    f"Balance strategy:     "
    f"{BALANCED_SAMPLER_STRATEGY if use_balanced_sampler else 'none'}"
)

# Sample expensive diagnostics so TensorBoard remains practical on large data.
previous_writer = globals().get("TENSORBOARD_WRITER")
if previous_writer is not None:
    previous_writer.close()

TENSORBOARD_WRITER = None
TENSORBOARD_WRITER_CLOSED = False
TENSORBOARD_RUN_DIR = None
if TENSORBOARD_ENABLED:
    resolved_run_name = TENSORBOARD_RUN_NAME.strip()
    if not resolved_run_name:
        resolved_run_name = f"ugvnet_{time.strftime('%Y%m%d_%H%M%S')}"
    TENSORBOARD_RUN_DIR = TENSORBOARD_ROOT / resolved_run_name
    TENSORBOARD_WRITER = SummaryWriter(
        log_dir=str(TENSORBOARD_RUN_DIR),
        max_queue=20,
        flush_secs=TENSORBOARD_FLUSH_SECONDS,
    )


def tensorboard_scalar(tag, value, step):
    if TENSORBOARD_WRITER is None or TENSORBOARD_WRITER_CLOSED:
        return
    TENSORBOARD_WRITER.add_scalar(tag, float(value), int(step))


def tensorboard_text(tag, value, step=0):
    if TENSORBOARD_WRITER is None or TENSORBOARD_WRITER_CLOSED:
        return
    TENSORBOARD_WRITER.add_text(tag, str(value), int(step))


def tensorboard_selected_parameters(module):
    """Select informative tensors without serializing both full backbones."""
    exact_names = {
        "fusion.efficientnet_projection.0.weight",
        "fusion.convnext_projection.0.weight",
        "fusion.gate.0.weight",
        "fusion.gate.2.weight",
        "head.weight",
        "head.bias",
    }
    selected = []
    backbone_candidates = {
        "efficientnet_features": [],
        "convnext_features": [],
    }
    for name, parameter in module.named_parameters():
        if name in exact_names or any(
            token in name
            for token in (
                ".attention.qkv.weight",
                ".attention.projection.weight",
                ".mlp.0.weight",
                ".mlp.5.weight",
            )
        ):
            selected.append((name, parameter))
        for prefix, candidates in backbone_candidates.items():
            if name.startswith(prefix) and name.endswith("weight") and parameter.ndim > 1:
                candidates.append((name, parameter))
    for candidates in backbone_candidates.values():
        if candidates:
            selected.append(candidates[-1])
    unique = {}
    for name, parameter in selected:
        unique[name] = parameter
    return list(unique.items())[:20]


def tensorboard_log_histograms(module, step, gradients=False):
    if TENSORBOARD_WRITER is None or TENSORBOARD_WRITER_CLOSED:
        return
    prefix = "Gradients" if gradients else "Parameters"
    for name, parameter in tensorboard_selected_parameters(module):
        values = parameter.grad if gradients else parameter
        if values is None or values.numel() == 0:
            continue
        clean_name = name.replace(".", "/")
        TENSORBOARD_WRITER.add_histogram(
            f"{prefix}/{clean_name}",
            values.detach().float().cpu(),
            int(step),
        )


def tensorboard_activation_modules(module):
    """Return a small, interpretable set of hybrid activations."""
    named_modules = dict(module.named_modules())
    requested = (
        "fusion.efficientnet_projection.2",
        "fusion.convnext_projection.2",
        "fusion.gate.2",
        "global_fusion.0.attention",
        "global_fusion.0.mlp.5",
        "final_norm",
        "pool",
        "head",
    )
    return [
        (name, named_modules[name])
        for name in requested
        if name in named_modules
    ]


def tensorboard_log_optimizer_histograms(optimizer, module, step):
    if (
        TENSORBOARD_WRITER is None
        or TENSORBOARD_WRITER_CLOSED
        or not TENSORBOARD_LOG_OPTIMIZER_STATE
    ):
        return
    selected = {
        id(parameter): name
        for name, parameter in tensorboard_selected_parameters(module)
    }
    for parameter, state in optimizer.state.items():
        name = selected.get(id(parameter))
        if name is None:
            continue
        clean_name = name.replace(".", "/")
        for state_name in ("exp_avg", "exp_avg_sq"):
            values = state.get(state_name)
            if values is None or values.numel() == 0:
                continue
            TENSORBOARD_WRITER.add_histogram(
                f"Optimizer/{state_name}/{clean_name}",
                values.detach().float().cpu(),
                int(step),
            )


def tensorboard_log_full_model_histograms(module, step):
    if (
        TENSORBOARD_WRITER is None
        or TENSORBOARD_WRITER_CLOSED
        or not TENSORBOARD_LOG_FULL_MODEL_HISTOGRAMS
    ):
        return
    for name, parameter in module.named_parameters():
        if parameter.numel() == 0:
            continue
        TENSORBOARD_WRITER.add_histogram(
            f"FullModel/{name.replace('.', '/')}",
            parameter.detach().float().cpu(),
            int(step),
        )


def _read_proc_key(path, key):
    try:
        for line in Path(path).read_text(encoding="utf-8").splitlines():
            if line.startswith(key):
                return float(line.split()[1])
    except (OSError, ValueError, IndexError):
        return None
    return None


def tensorboard_log_system_metrics(step):
    if (
        TENSORBOARD_WRITER is None
        or TENSORBOARD_WRITER_CLOSED
        or not TENSORBOARD_LOG_SYSTEM_METRICS
    ):
        return

    process_rss_kb = _read_proc_key("/proc/self/status", "VmRSS:")
    memory_total_kb = _read_proc_key("/proc/meminfo", "MemTotal:")
    memory_available_kb = _read_proc_key("/proc/meminfo", "MemAvailable:")
    if process_rss_kb is not None:
        tensorboard_scalar("System/process_rss_gb", process_rss_kb / 2**20, step)
    if memory_total_kb and memory_available_kb is not None:
        used_percent = 100 * (1 - memory_available_kb / memory_total_kb)
        tensorboard_scalar("System/host_ram_used_percent", used_percent, step)

    try:
        load_average = float(Path("/proc/loadavg").read_text().split()[0])
        tensorboard_scalar(
            "System/cpu_load_percent",
            100 * load_average / max(1, os.cpu_count() or 1),
            step,
        )
    except (OSError, ValueError, IndexError):
        pass

    try:
        disk = shutil.disk_usage(PROJECT_OUTPUT_ROOT)
        tensorboard_scalar("System/disk_used_percent", 100 * disk.used / disk.total, step)
        tensorboard_scalar("System/disk_free_gb", disk.free / 2**30, step)
    except OSError:
        pass

    for proc_key, tag in (
        ("read_bytes:", "System/process_read_gb"),
        ("write_bytes:", "System/process_write_gb"),
    ):
        value = _read_proc_key("/proc/self/io", proc_key)
        if value is not None:
            tensorboard_scalar(tag, value / 2**30, step)

    if DEVICE.type == "cuda":
        tensorboard_scalar(
            "System/gpu_memory_allocated_gb",
            torch.cuda.memory_allocated() / 2**30,
            step,
        )
        tensorboard_scalar(
            "System/gpu_memory_reserved_gb",
            torch.cuda.memory_reserved() / 2**30,
            step,
        )
        tensorboard_scalar(
            "System/gpu_peak_memory_allocated_gb",
            torch.cuda.max_memory_allocated() / 2**30,
            step,
        )
        try:
            completed = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw",
                    "--format=csv,noheader,nounits",
                ],
                capture_output=True,
                text=True,
                timeout=5,
                check=False,
            )
            if completed.returncode == 0:
                raw_values = completed.stdout.splitlines()[0].split(",")
                for tag, raw_value in zip(
                    (
                        "System/gpu_utilization_percent",
                        "System/nvidia_memory_used_mb",
                        "System/nvidia_memory_total_mb",
                        "System/gpu_temperature_c",
                        "System/gpu_power_watts",
                    ),
                    raw_values,
                ):
                    try:
                        tensorboard_scalar(tag, float(raw_value.strip()), step)
                    except ValueError:
                        continue
        except (OSError, subprocess.SubprocessError, ValueError, IndexError):
            pass
        torch.cuda.reset_peak_memory_stats()


def _tensorboard_shape_text(value):
    if isinstance(value, torch.Tensor):
        return " x ".join(str(size) for size in value.shape)
    if isinstance(value, (list, tuple)):
        parts = [_tensorboard_shape_text(item) for item in value]
        return " + ".join(part for part in parts if part)[:120]
    return ""


def _compact_parameter_count(count):
    if count >= 1_000_000:
        return f"{count / 1_000_000:.2f}M"
    if count >= 1_000:
        return f"{count / 1_000:.1f}K"
    return str(count)


def _capture_architecture_shapes(module, sample):
    """Capture the real output shape of every displayed architecture stage."""
    shapes = {}
    handles = []

    def register(name, layer):
        def capture(_layer, _inputs, output):
            shapes[name] = _tensorboard_shape_text(output)

        handles.append(layer.register_forward_hook(capture))

    for index, layer in enumerate(module.efficientnet_features):
        register(f"efficientnet.{index}", layer)
    for index, layer in enumerate(module.convnext_features):
        register(f"convnext.{index}", layer)
    register("fusion.efficientnet_projection", module.fusion.efficientnet_projection)
    register("fusion.convnext_projection", module.fusion.convnext_projection)
    register("fusion.gate", module.fusion.gate)
    register("fusion.output", module.fusion)
    for index, layer in enumerate(module.global_fusion):
        register(f"global_fusion.{index}", layer)
    register("final_norm", module.final_norm)
    register("pool", module.pool)
    register("dropout", module.dropout)
    register("head", module.head)

    try:
        with torch.inference_mode():
            module(sample)
    finally:
        for handle in handles:
            handle.remove()
    return shapes


def tensorboard_architecture_figure(module, sample):
    """Build a readable, stage-complete map of the executed hybrid model."""
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

    shapes = _capture_architecture_shapes(module, sample)
    figure, axis = plt.subplots(figsize=(20, 14), facecolor="#08111f")
    axis.set_facecolor("#08111f")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.axis("off")

    def box(x, y, width, height, text, color, size=8, edge="#dbeafe"):
        patch = FancyBboxPatch(
            (x, y),
            width,
            height,
            boxstyle="round,pad=0.006,rounding_size=0.008",
            linewidth=1.2,
            edgecolor=edge,
            facecolor=color,
        )
        axis.add_patch(patch)
        axis.text(
            x + width / 2,
            y + height / 2,
            text,
            ha="center",
            va="center",
            color="white",
            fontsize=size,
            family="DejaVu Sans",
            linespacing=1.22,
        )
        return (x, y, width, height)

    def arrow(start, end, color="#94a3b8"):
        axis.add_patch(FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=12,
            linewidth=1.25,
            color=color,
            connectionstyle="arc3,rad=0",
        ))

    def stage_text(prefix, index, layer):
        parameters = sum(parameter.numel() for parameter in layer.parameters())
        blocks = len(layer) if isinstance(layer, nn.Sequential) else 1
        block_text = f"{blocks} block{'s' if blocks != 1 else ''}"
        return (
            f"Stage {index + 1}: {type(layer).__name__}\n"
            f"{block_text} | {_compact_parameter_count(parameters)} params\n"
            f"output: {shapes.get(f'{prefix}.{index}', 'not captured')}"
        )

    total = sum(parameter.numel() for parameter in module.parameters())
    trainable = sum(
        parameter.numel() for parameter in module.parameters() if parameter.requires_grad
    )
    axis.text(
        0.5,
        0.975,
        "UGVNet - Unified Global Vision Network",
        ha="center",
        va="center",
        color="white",
        fontsize=21,
        weight="bold",
        family="DejaVu Sans",
    )
    axis.text(
        0.5,
        0.948,
        (
            "EfficientNetV2-S + ConvNeXt-Tiny | adaptive gated fusion + global attention | "
            f"{_compact_parameter_count(total)} total / "
            f"{_compact_parameter_count(trainable)} currently trainable parameters"
        ),
        ha="center",
        va="center",
        color="#cbd5e1",
        fontsize=10,
        family="DejaVu Sans",
    )

    input_box = box(
        0.39,
        0.892,
        0.22,
        0.038,
        f"INPUT IMAGE\n1 x 3 x {IMAGE_SIZE} x {IMAGE_SIZE}",
        "#0f766e",
        size=9,
    )
    axis.text(0.245, 0.865, "EfficientNetV2-S branch", color="#7dd3fc", fontsize=12,
              ha="center", weight="bold")
    axis.text(0.755, 0.865, "ConvNeXt-Tiny branch", color="#c4b5fd", fontsize=12,
              ha="center", weight="bold")

    left_layers = list(module.efficientnet_features)
    right_layers = list(module.convnext_features)
    row_count = max(len(left_layers), len(right_layers))
    stage_top = 0.825
    stage_bottom = 0.455
    stage_gap = 0.007
    stage_height = (stage_top - stage_bottom - stage_gap * (row_count - 1)) / row_count
    left_boxes = []
    right_boxes = []
    for index, layer in enumerate(left_layers):
        y = stage_top - (index + 1) * stage_height - index * stage_gap
        left_boxes.append(box(
            0.045, y, 0.40, stage_height,
            stage_text("efficientnet", index, layer), "#075985", size=7.4,
        ))
    for index, layer in enumerate(right_layers):
        y = stage_top - (index + 1) * stage_height - index * stage_gap
        right_boxes.append(box(
            0.555, y, 0.40, stage_height,
            stage_text("convnext", index, layer), "#5b21b6", size=7.4,
        ))

    arrow((0.45, input_box[1]), (0.245, left_boxes[0][1] + left_boxes[0][3]))
    arrow((0.55, input_box[1]), (0.755, right_boxes[0][1] + right_boxes[0][3]))
    for items in (left_boxes, right_boxes):
        for upper, lower in itertools.pairwise(items):
            arrow(
                (upper[0] + upper[2] / 2, upper[1]),
                (lower[0] + lower[2] / 2, lower[1] + lower[3]),
            )

    fusion_shape = shapes.get("fusion.output", "not captured").split(" + ")[0]
    fusion_box = box(
        0.12,
        0.335,
        0.76,
        0.085,
        (
            "ADAPTIVE BACKBONE FUSION\n"
            f"EfficientNet 1x1 projection: {shapes.get('fusion.efficientnet_projection', '?')}   |   "
            f"ConvNeXt 1x1 projection: {shapes.get('fusion.convnext_projection', '?')}\n"
            "spatial alignment -> concatenate -> learned 1x1 gate -> two-way softmax -> weighted sum\n"
            f"fused feature map: {fusion_shape}"
        ),
        "#9a3412",
        size=8.4,
        edge="#fed7aa",
    )
    arrow(
        (left_boxes[-1][0] + left_boxes[-1][2] / 2, left_boxes[-1][1]),
        (0.36, fusion_box[1] + fusion_box[3]),
        "#38bdf8",
    )
    arrow(
        (right_boxes[-1][0] + right_boxes[-1][2] / 2, right_boxes[-1][1]),
        (0.64, fusion_box[1] + fusion_box[3]),
        "#a78bfa",
    )

    global_lines = []
    for index, layer in enumerate(module.global_fusion):
        parameters = sum(parameter.numel() for parameter in layer.parameters())
        global_lines.append(
            f"Block {index + 1}: position DWConv -> LN + {ATTENTION_HEADS}-head global self-attention "
            f"-> LN + 4x convolutional MLP -> LayerScale residuals | "
            f"{_compact_parameter_count(parameters)} params | {shapes.get(f'global_fusion.{index}', '?')}"
        )
    global_box = box(
        0.10,
        0.225,
        0.80,
        0.075,
        "GLOBAL FUSION REFINEMENT\n" + "\n".join(global_lines),
        "#9f1239",
        size=8.0,
        edge="#fecdd3",
    )
    arrow((0.5, fusion_box[1]), (0.5, global_box[1] + global_box[3]))

    classifier_box = box(
        0.13,
        0.105,
        0.74,
        0.080,
        (
            "CLASSIFICATION PIPELINE\n"
            f"LayerNorm2d: {shapes.get('final_norm', '?')} -> "
            f"AdaptiveAvgPool2d(1): {shapes.get('pool', '?')} -> flatten\n"
            f"Dropout(p={current_dropout:.2f}) -> Linear({FUSION_CHANNELS}, {NUM_CLASSES}): "
            f"{shapes.get('head', '?')} -> class logits"
        ),
        "#166534",
        size=8.6,
        edge="#bbf7d0",
    )
    arrow((0.5, global_box[1]), (0.5, classifier_box[1] + classifier_box[3]))
    output_box = box(
        0.36,
        0.035,
        0.28,
        0.038,
        f"OUTPUT\n{NUM_CLASSES} class scores",
        "#0f766e",
        size=9,
    )
    arrow((0.5, classifier_box[1]), (0.5, output_box[1] + output_box[3]))
    figure.tight_layout(pad=0.5)
    return figure


def tensorboard_log_expandable_graph(module, sample):
    """Write the real traced graph with the single outer scope promoted."""
    import warnings as warning_tools

    from torch.utils.tensorboard._pytorch_graph import graph as build_graph

    # The spatial-alignment condition is fixed for this sample shape and safe to trace.
    with warning_tools.catch_warnings(), torch.inference_mode():
        warning_tools.filterwarnings("ignore", category=torch.jit.TracerWarning)
        graph_definition, run_metadata = build_graph(
            module,
            sample,
            verbose=False,
            use_strict_trace=False,
        )
    root_prefix = f"{type(module).__name__}/"
    friendly_scopes = (
        ("Sequential[efficientnet_features]/", "01 EfficientNetV2-S Backbone/"),
        ("Sequential[convnext_features]/", "02 ConvNeXt-Tiny Backbone/"),
        ("AdaptiveBackboneFusion[fusion]/", "03 Adaptive Backbone Fusion/"),
        ("Sequential[global_fusion]/", "04 Global Fusion Refinement/"),
        ("LayerNorm2d[final_norm]/", "05 Final Normalization/"),
        ("AdaptiveAvgPool2d[pool]/", "06 Global Average Pool/"),
        ("Dropout[dropout]/", "07 Dropout/"),
        ("Linear[head]/", "08 Classification Head/"),
    )

    def friendly_name(value):
        control_prefix = "^" if value.startswith("^") else ""
        core = value[1:] if control_prefix else value
        core = core.removeprefix(root_prefix)
        for original, replacement in friendly_scopes:
            if core.startswith(original):
                core = replacement + core[len(original):]
                break
        return control_prefix + core

    for node in graph_definition.node:
        node.name = friendly_name(node.name)
        for index, input_name in enumerate(node.input):
            node.input[index] = friendly_name(input_name)

    TENSORBOARD_WRITER._get_file_writer().add_graph((graph_definition, run_metadata))
    return len(graph_definition.node)


if TENSORBOARD_WRITER is not None:
    run_configuration = {
        "platform": PLATFORM,
        "training_mode": RESOLVED_MODE,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "classes": NUM_CLASSES,
        "fusion_channels": FUSION_CHANNELS,
        "fusion_depth": FUSION_DEPTH,
        "attention_heads": ATTENTION_HEADS,
        "backbone_learning_rate": BACKBONE_LR,
        "fusion_learning_rate": FUSION_LR,
        "weight_decay": WEIGHT_DECAY,
        "dropout": DROPOUT,
        "label_smoothing": LABEL_SMOOTHING,
        "mixed_precision": AMP_ENABLED,
        "fused_adamw": FUSED_ADAMW_ENABLED,
        "seed": SEED,
        "loss_function": RESOLVED_LOSS_FUNCTION,
        "focal_gamma": current_focal_gamma if use_focal_loss else None,
        "gap_epoch_rejection": False,
        "gap_adaptation_threshold": GENERALIZATION_GAP_THRESHOLD,
        "gap_adaptation_patience": GAP_ADAPTATION_PATIENCE,
        "gap_adaptation_lr_factor": GAP_ADAPTATION_LR_FACTOR,
        "gap_adaptation_dropout_step": GAP_ADAPTATION_DROPOUT_STEP,
        "gap_adaptation_label_smoothing_step": (
            GAP_ADAPTATION_LABEL_SMOOTHING_STEP
        ),
        "gap_adaptation_focal_gamma_step": GAP_ADAPTATION_FOCAL_GAMMA_STEP,
        "balanced_sampler_strategy": (
            BALANCED_SAMPLER_STRATEGY if use_balanced_sampler else "none"
        ),
        "effective_number_beta": resolved_sampler_beta,
        "minority_augmentation_classes": [
            CLASSES[index] for index in sorted(MINORITY_CLASS_INDICES)
        ],
        "resumed_from_checkpoint": bool(resume_checkpoint),
        "histogram_interval": TENSORBOARD_HISTOGRAM_INTERVAL,
        "profile_batches": TENSORBOARD_PROFILE_BATCHES,
        "activation_interval": TENSORBOARD_ACTIVATION_INTERVAL,
        "optimizer_interval": TENSORBOARD_OPTIMIZER_INTERVAL,
        "embedding_interval": TENSORBOARD_EMBEDDING_INTERVAL,
        "class_metrics": TENSORBOARD_LOG_CLASS_METRICS,
        "system_metrics": TENSORBOARD_LOG_SYSTEM_METRICS,
        "roc_curves": TENSORBOARD_LOG_ROC_CURVES,
        "calibration": TENSORBOARD_LOG_CALIBRATION,
        "gradcam": TENSORBOARD_LOG_GRADCAM,
        "full_model_histograms": TENSORBOARD_LOG_FULL_MODEL_HISTOGRAMS,
        "architecture_image": TENSORBOARD_LOG_ARCHITECTURE_IMAGE,
        "expandable_operation_graph": TENSORBOARD_LOG_GRAPH,
    }
    tensorboard_text(
        "Run/configuration",
        "```json\n" + json.dumps(run_configuration, indent=2) + "\n```",
    )
    tensorboard_text("Run/classes", ", ".join(CLASSES))
    TENSORBOARD_WRITER.add_custom_scalars({
        "UGVNet accuracy": {
            "Train, validation, and test": [
                "Multiline",
                [
                    "Epoch/train/accuracy",
                    "Epoch/validation/accuracy",
                    "Test/accuracy",
                ],
            ],
        },
        "UGVNet quality": {
            "Macro F1": [
                "Multiline",
                [
                    "Epoch/train/macro_f1",
                    "Epoch/validation/macro_f1",
                    "Test/macro_f1",
                ],
            ],
            "Generalization gap": [
                "Multiline",
                ["Epoch/generalization_gap"],
            ],
        },
        "UGVNet throughput": {
            "Images per second": [
                "Multiline",
                [
                    "Epoch/train/images_per_second",
                    "Epoch/validation/images_per_second",
                    "Test/images_per_second",
                ],
            ],
        },
    })
    tensorboard_scalar("Model/total_parameters", total_parameters, 0)
    tensorboard_scalar("Model/initially_trainable_parameters", trainable_parameters, 0)
    tensorboard_scalar(
        "Run/generalization_gap_threshold",
        GENERALIZATION_GAP_THRESHOLD,
        0,
    )
    if DATASET_AUDIT is not None:
        for key, value in DATASET_AUDIT["summary"].items():
            if isinstance(value, (int, float)):
                tensorboard_scalar(f"Audit/{key}", value, 0)
        tensorboard_scalar(
            "Audit/duration_seconds",
            DATASET_AUDIT["duration_seconds"],
            0,
        )
        tensorboard_scalar(
            "Audit/training_imbalance_ratio",
            DATASET_AUDIT["training_imbalance_ratio"],
            0,
        )
    if TENSORBOARD_LOG_GRAPH or TENSORBOARD_LOG_ARCHITECTURE_IMAGE:
        graph_input = torch.zeros(
            1,
            3,
            IMAGE_SIZE,
            IMAGE_SIZE,
            device=DEVICE,
        )
        if CHANNELS_LAST_ENABLED:
            graph_input = graph_input.to(memory_format=torch.channels_last)
        graph_training_state = model.training
        model.eval()
        try:
            if TENSORBOARD_LOG_ARCHITECTURE_IMAGE:
                try:
                    architecture_figure = tensorboard_architecture_figure(
                        model,
                        graph_input,
                    )
                    architecture_path = RESULTS_DIR / "ugvnet_full_architecture.png"
                    architecture_figure.savefig(
                        architecture_path,
                        dpi=180,
                        bbox_inches="tight",
                        facecolor=architecture_figure.get_facecolor(),
                    )
                    TENSORBOARD_WRITER.add_figure(
                        "Model/full_architecture",
                        architecture_figure,
                        global_step=0,
                        close=True,
                    )
                    tensorboard_text(
                        "Model/architecture_guide",
                        (
                            "The Graphs dashboard contains the complete executed operation graph. "
                            "Its outer UGVNet scope has been promoted into named backbone, fusion, "
                            "attention, normalization, pooling, dropout, and classifier sections. "
                            "Click any section to inspect every traced operation. The Images dashboard "
                            "contains the readable stage-level map with real shapes and parameter counts."
                        ),
                    )
                    print(f"TensorBoard architecture image: {architecture_path}")
                except Exception as image_error:  # noqa: BLE001
                    print(
                        "TensorBoard architecture image warning: "
                        f"{type(image_error).__name__}: {image_error}"
                    )
            if TENSORBOARD_LOG_GRAPH:
                try:
                    graph_node_count = tensorboard_log_expandable_graph(
                        model,
                        graph_input,
                    )
                    print(
                        "TensorBoard expandable operation graph: "
                        f"logged ({graph_node_count:,} traced nodes)"
                    )
                except Exception as trace_error:  # noqa: BLE001
                    print(
                        "TensorBoard operation graph warning: "
                        f"{type(trace_error).__name__}: {trace_error}"
                    )
        finally:
            model.train(graph_training_state)
            del graph_input
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
    tensorboard_log_histograms(model, 0, gradients=False)
    tensorboard_log_system_metrics(0)
    TENSORBOARD_WRITER.flush()
    print("TensorBoard initial parameter histograms: logged")
    print(f"TensorBoard run:      {TENSORBOARD_RUN_DIR}")
else:
    print("TensorBoard:          disabled")



## 14. Open the TensorBoard dashboard

Launch the dashboard before training for live scalar, fusion, class-metric,
system, activation, optimizer, embedding, and profiler updates. After Cell 13,
Graphs shows the real operation graph divided into named expandable architecture
sections. Images > `Model/full_architecture` shows the complete readable map with
executed shapes and parameter counts. Refresh TensorBoard after Cell 19 to load
ROC, calibration, Grad-CAM, final PR curves, Projector, and HParams.


In [ ]:
# 14. TensorBoard dashboard
if not TENSORBOARD_ENABLED:
    print("TensorBoard is disabled in Cell 04.")
else:
    notebook_shell = get_ipython()
    if notebook_shell is None:
        raise RuntimeError("TensorBoard dashboard requires a notebook runtime.")
    print(f"TensorBoard log root: {TENSORBOARD_ROOT}")
    print(f"Current run: {TENSORBOARD_RUN_DIR}")
    dashboards = (
        "Scalars, Text, Graphs, Histograms, Images, HParams, "
        "PR/ROC/Calibration, Projector, Grad-CAM, and system telemetry"
    )
    if TENSORBOARD_PROFILER_AVAILABLE:
        dashboards += ", and Profiler"
    else:
        dashboards += " (Profiler trace recorded; UI plugin unavailable)"
    print(f"Dashboards: {dashboards}")
    notebook_shell.run_line_magic("load_ext", "tensorboard")
    notebook_shell.run_line_magic(
        "tensorboard",
        f"--logdir {TENSORBOARD_ROOT} --port 6006 --reload_interval 5",
    )


## 15. Metrics

**This cell covers:** Confusion-matrix accumulation and macro-F1 calculation. Macro-F1 gives equal importance to minority and majority disease classes.

In [ ]:
# 15. Metrics
def safe_tensorboard_name(value):
    cleaned = "".join(
        character if character.isalnum() else "_"
        for character in str(value)
    ).strip("_")
    return cleaned or "unnamed"


def class_metrics_from_confusion(confusion):
    confusion = confusion.float()
    true_positive = confusion.diag()
    false_positive = confusion.sum(0) - true_positive
    false_negative = confusion.sum(1) - true_positive
    true_negative = confusion.sum() - true_positive - false_positive - false_negative

    def safe_divide(numerator, denominator):
        return torch.where(
            denominator > 0,
            numerator / denominator,
            torch.zeros_like(denominator),
        )

    precision = safe_divide(true_positive, true_positive + false_positive)
    recall = safe_divide(true_positive, true_positive + false_negative)
    specificity = safe_divide(true_negative, true_negative + false_positive)
    f1 = safe_divide(2 * true_positive, 2 * true_positive + false_positive + false_negative)
    support = confusion.sum(1)
    return {
        "class_precision": precision.tolist(),
        "class_recall": recall.tolist(),
        "class_specificity": specificity.tolist(),
        "class_f1": f1.tolist(),
        "class_support": support.tolist(),
        "macro_precision": precision.mean().item(),
        "macro_recall": recall.mean().item(),
        "macro_specificity": specificity.mean().item(),
        "macro_f1": f1.mean().item(),
    }


def macro_f1_from_confusion(confusion):
    metrics = class_metrics_from_confusion(confusion)
    return metrics["macro_f1"], metrics["class_f1"]


def binary_roc_curve(targets, scores):
    targets = targets.detach().cpu().bool()
    scores = scores.detach().cpu().float()
    positive_count = int(targets.sum())
    negative_count = int((~targets).sum())
    if positive_count == 0 or negative_count == 0:
        return None, None, None

    order = torch.argsort(scores, descending=True)
    sorted_targets = targets[order]
    sorted_scores = scores[order]
    cumulative_true = sorted_targets.long().cumsum(0)
    cumulative_false = (~sorted_targets).long().cumsum(0)
    distinct = torch.where(sorted_scores[1:] != sorted_scores[:-1])[0]
    threshold_indices = torch.cat(
        (distinct, torch.tensor([sorted_scores.numel() - 1]))
    )
    true_positive_rate = cumulative_true[threshold_indices].float() / positive_count
    false_positive_rate = cumulative_false[threshold_indices].float() / negative_count
    true_positive_rate = torch.cat((torch.zeros(1), true_positive_rate))
    false_positive_rate = torch.cat((torch.zeros(1), false_positive_rate))
    auc = torch.trapz(true_positive_rate, false_positive_rate).item()
    return false_positive_rate.numpy(), true_positive_rate.numpy(), auc


def reliability_curve(targets, probabilities, bins=15):
    confidences, predictions = probabilities.max(1)
    correctness = predictions.eq(targets).float()
    boundaries = torch.linspace(0, 1, bins + 1)
    bin_confidence = []
    bin_accuracy = []
    bin_fraction = []
    for bin_index in range(bins):
        lower = boundaries[bin_index]
        upper = boundaries[bin_index + 1]
        mask = (confidences > lower) & (confidences <= upper)
        if bin_index == 0:
            mask = (confidences >= lower) & (confidences <= upper)
        if mask.any():
            bin_confidence.append(confidences[mask].mean().item())
            bin_accuracy.append(correctness[mask].mean().item())
            bin_fraction.append(mask.float().mean().item())
    return bin_confidence, bin_accuracy, bin_fraction


def probability_metrics(targets, probabilities):
    class_auc = []
    for class_index in range(probabilities.size(1)):
        _, _, auc = binary_roc_curve(
            targets == class_index,
            probabilities[:, class_index],
        )
        class_auc.append(auc)
    valid_auc = [value for value in class_auc if value is not None]
    bin_confidence, bin_accuracy, bin_fraction = reliability_curve(
        targets,
        probabilities,
    )
    expected_calibration_error = sum(
        abs(accuracy - confidence) * fraction
        for confidence, accuracy, fraction in zip(
            bin_confidence,
            bin_accuracy,
            bin_fraction,
        )
    )
    one_hot_targets = F.one_hot(
        targets,
        num_classes=probabilities.size(1),
    ).float()
    brier_score = ((probabilities - one_hot_targets) ** 2).sum(1).mean().item()
    return {
        "class_roc_auc": class_auc,
        "macro_roc_auc": sum(valid_auc) / len(valid_auc) if valid_auc else None,
        "expected_calibration_error": expected_calibration_error,
        "multiclass_brier_score": brier_score,
    }


def tensorboard_log_class_metrics(prefix, metrics, step):
    for aggregate_name in (
        "macro_precision",
        "macro_recall",
        "macro_specificity",
        "macro_roc_auc",
        "expected_calibration_error",
        "multiclass_brier_score",
    ):
        value = metrics.get(aggregate_name)
        if value is not None:
            tensorboard_scalar(f"{prefix}/{aggregate_name}", value, step)

    for metric_name in (
        "class_precision",
        "class_recall",
        "class_specificity",
        "class_f1",
        "class_roc_auc",
    ):
        for class_index, value in enumerate(metrics.get(metric_name, [])):
            if value is None:
                continue
            class_tag = safe_tensorboard_name(CLASSES[class_index])
            tensorboard_scalar(
                f"{prefix}/{metric_name}/{class_index:02d}_{class_tag}",
                value,
                step,
            )


def format_metrics(metrics):
    return (
        f"loss={metrics['loss']:.4f} | "
        f"accuracy={metrics['accuracy']:.4f} | "
        f"macro_f1={metrics['macro_f1']:.4f}"
    )


## 16. Run one epoch

**This cell covers:** Mixed-precision training/evaluation, GPU-side confusion
metrics, probability-based ROC-AUC and calibration, hybrid diagnostics,
sampled activation hooks, optimizer gradients, validation embedding capture,
system-aware profiling, and throughput reporting.


In [ ]:
# 16. Epoch loop
def run_epoch(
    model,
    loader,
    training=False,
    split_name="dataset",
    collect_tensorboard=False,
    collect_outputs=False,
    collect_curve_metrics=False,
    gradient_histogram_step=None,
    activation_histogram_step=None,
    profile_batches=0,
    max_prediction_images=0,
):
    if collect_outputs and not collect_tensorboard:
        raise ValueError("collect_outputs requires collect_tensorboard=True")

    model.train(training)
    started = time.time()
    total_loss = torch.zeros((), device=DEVICE)
    total_correct = torch.zeros((), dtype=torch.long, device=DEVICE)
    total_gradient_norm = torch.zeros((), device=DEVICE)
    gradient_steps = 0
    total_examples = 0
    confusion = torch.zeros(
        NUM_CLASSES,
        NUM_CLASSES,
        dtype=torch.long,
        device=DEVICE,
    )
    hybrid_sums = {}
    attention_entropy_sum = torch.zeros((), device=DEVICE)
    attention_entropy_examples = 0
    output_probabilities = []
    output_targets = []
    output_embeddings = []
    sample_images = []
    sample_targets = []
    sample_predictions = []
    sample_confidences = []
    fusion_weight_samples = []
    activation_handles = []
    captured_activations = set()
    if (
        activation_histogram_step is not None
        and TENSORBOARD_WRITER is not None
        and not TENSORBOARD_WRITER_CLOSED
        and TENSORBOARD_LOG_ACTIVATIONS
    ):
        def make_activation_hook(module_name):
            def capture_activation(_module, _inputs, output):
                if module_name in captured_activations:
                    return
                values = output
                if isinstance(values, (tuple, list)):
                    values = next(
                        (item for item in values if torch.is_tensor(item)),
                        None,
                    )
                if not torch.is_tensor(values) or values.numel() == 0:
                    return
                captured_activations.add(module_name)
                TENSORBOARD_WRITER.add_histogram(
                    f"Activations/{module_name.replace('.', '/')}",
                    values.detach().float().cpu(),
                    int(activation_histogram_step),
                )
            return capture_activation

        for activation_name, activation_module in tensorboard_activation_modules(model):
            activation_handles.append(
                activation_module.register_forward_hook(
                    make_activation_hook(activation_name)
                )
            )

    model.fusion.capture_tensorboard_stats = collect_tensorboard
    attention_modules = [
        module for module in model.modules()
        if isinstance(module, GlobalSelfAttention)
    ]
    for module in attention_modules:
        module.capture_tensorboard_stats = collect_tensorboard

    should_profile = (
        training
        and profile_batches > 0
        and TENSORBOARD_RUN_DIR is not None
    )
    if should_profile:
        activities = [ProfilerActivity.CPU]
        if DEVICE.type == "cuda":
            activities.append(ProfilerActivity.CUDA)
        profiler_context = profile(
            activities=activities,
            schedule=schedule(
                wait=1,
                warmup=1,
                active=max(1, profile_batches - 2),
                repeat=1,
            ),
            on_trace_ready=tensorboard_trace_handler(
                str(TENSORBOARD_RUN_DIR)
            ),
            record_shapes=True,
            profile_memory=True,
            with_stack=False,
            acc_events=True,
        )
    else:
        profiler_context = nullcontext(None)

    gradient_context = torch.enable_grad if training else torch.inference_mode
    progress_interval = max(1, len(loader) // 5)

    with gradient_context():  # noqa: SIM117
        with profiler_context as active_profiler:
            for batch_index, (images, targets) in enumerate(loader, start=1):
                images = images.to(DEVICE, non_blocking=True)
                if CHANNELS_LAST_ENABLED:
                    images = images.to(memory_format=torch.channels_last)
                targets = targets.to(DEVICE, non_blocking=True)
                if training:
                    optimizer.zero_grad(set_to_none=True)

                with torch.autocast(
                    device_type=DEVICE.type,
                    dtype=torch.float16,
                    enabled=AMP_ENABLED,
                ):
                    if collect_tensorboard:
                        features, fusion_weights = model.forward_features(
                            images,
                            return_fusion_weights=True,
                        )
                        embeddings = model.pool(features).flatten(1)
                        logits = model.head(model.dropout(embeddings))
                    else:
                        fusion_weights = None
                        embeddings = None
                        logits = model(images)
                    loss = criterion(logits, targets)

                if training:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    gradient_norm = nn.utils.clip_grad_norm_(
                        model.parameters(),
                        GRADIENT_CLIP,
                    )
                    total_gradient_norm += gradient_norm.detach()
                    gradient_steps += 1
                    if (
                        gradient_histogram_step is not None
                        and batch_index == len(loader)
                    ):
                        tensorboard_log_histograms(
                            model,
                            gradient_histogram_step,
                            gradients=True,
                        )
                    scaler.step(optimizer)
                    scaler.update()

                predictions = logits.argmax(1)
                batch_size = targets.size(0)
                total_loss += loss.detach() * batch_size
                total_correct += (predictions == targets).sum()
                total_examples += batch_size
                encoded = targets * NUM_CLASSES + predictions
                confusion += torch.bincount(
                    encoded,
                    minlength=NUM_CLASSES * NUM_CLASSES,
                ).reshape(NUM_CLASSES, NUM_CLASSES)

                if collect_tensorboard:
                    for key, value in model.fusion.last_tensorboard_stats.items():
                        hybrid_sums.setdefault(
                            key,
                            torch.zeros((), device=DEVICE),
                        )
                        hybrid_sums[key] += value.to(DEVICE) * batch_size
                    attention_values = [
                        module.last_attention_entropy
                        for module in attention_modules
                        if module.last_attention_entropy is not None
                    ]
                    if attention_values:
                        batch_attention_entropy = torch.stack(
                            [value.to(DEVICE) for value in attention_values]
                        ).mean()
                        attention_entropy_sum += batch_attention_entropy * batch_size
                        attention_entropy_examples += batch_size

                if collect_curve_metrics or collect_outputs:
                    probabilities = logits.detach().float().softmax(dim=1).cpu()
                    output_probabilities.append(probabilities)
                    output_targets.append(targets.detach().cpu())
                if collect_outputs:
                    output_embeddings.append(embeddings.detach().float().cpu())
                    remaining = max_prediction_images - sum(
                        tensor.size(0) for tensor in sample_images
                    )
                    if remaining > 0:
                        take = min(remaining, batch_size)
                        sample_images.append(images[:take].detach().float().cpu())
                        sample_targets.append(targets[:take].detach().cpu())
                        sample_predictions.append(predictions[:take].detach().cpu())
                        sample_confidences.append(
                            probabilities[:take].max(dim=1).values
                        )
                        if fusion_weights is not None:
                            fusion_weight_samples.append(
                                fusion_weights[:take].detach().float().cpu()
                            )

                if batch_index == 1 and activation_handles:
                    for handle in activation_handles:
                        handle.remove()
                    activation_handles.clear()

                if active_profiler is not None:
                    active_profiler.step()

                if batch_index % progress_interval == 0 or batch_index == len(loader):
                    elapsed = max(time.time() - started, 1e-6)
                    seen = min(total_examples, len(loader.dataset))
                    print(
                        f"\r{split_name:10s} {batch_index:4d}/{len(loader):4d} "
                        f"| {seen / elapsed:7.1f} images/s",
                        end="",
                    )

    for handle in activation_handles:
        handle.remove()
    model.fusion.capture_tensorboard_stats = False
    for module in attention_modules:
        module.capture_tensorboard_stats = False

    print()
    confusion_cpu = confusion.cpu()
    class_metrics = class_metrics_from_confusion(confusion_cpu)
    elapsed = max(time.time() - started, 1e-6)
    result = {
        "loss": (total_loss / total_examples).item(),
        "accuracy": (total_correct.float() / total_examples).item(),
        "confusion": confusion_cpu.tolist(),
        "images_per_second": total_examples / elapsed,
        **class_metrics,
    }
    if collect_curve_metrics:
        result.update(
            probability_metrics(
                torch.cat(output_targets),
                torch.cat(output_probabilities),
            )
        )
    if training and gradient_steps:
        result["gradient_norm"] = (
            total_gradient_norm / gradient_steps
        ).item()
    if collect_tensorboard:
        diagnostics = {
            key: (value / total_examples).item()
            for key, value in hybrid_sums.items()
        }
        if attention_entropy_examples:
            diagnostics["attention_entropy"] = (
                attention_entropy_sum / attention_entropy_examples
            ).item()
        result["hybrid_diagnostics"] = diagnostics
    if collect_outputs:
        result["_tensorboard_outputs"] = {
            "probabilities": torch.cat(output_probabilities),
            "targets": torch.cat(output_targets),
            "embeddings": torch.cat(output_embeddings),
            "sample_images": torch.cat(sample_images) if sample_images else None,
            "sample_targets": torch.cat(sample_targets) if sample_targets else None,
            "sample_predictions": (
                torch.cat(sample_predictions) if sample_predictions else None
            ),
            "sample_confidences": (
                torch.cat(sample_confidences) if sample_confidences else None
            ),
            "fusion_weights": (
                torch.cat(fusion_weight_samples)
                if fusion_weight_samples
                else None
            ),
        }
    return result


## 17. Train the model

**This cell covers:** Frozen-backbone warm-up, full fine-tuning, validation, cosine scheduling, macro-F1 checkpoint selection, early stopping, and non-blocking generalization-gap adaptation.

Every completed epoch is kept and advances the scheduler, history, patience, and checkpoint logic. A gap above `GENERALIZATION_GAP_THRESHOLD` never rejects or rolls back an epoch. If the gap remains high for `GAP_ADAPTATION_PATIENCE` consecutive epochs, the next epoch uses a gentler learning rate, more dropout and label smoothing, and—when focal loss is active—a lower focal gamma.

Each checkpoint stores the controller, early-stopping, optimizer, scaler, freeze, history, and portable random-number state for faithful continuation.


In [ ]:
# 17. Training loop
best_checkpoint_path = BEST_MODELS_DIR / "ugvnet_hybrid_best.pt"
last_checkpoint_path = CHECKPOINTS_DIR / "ugvnet_hybrid_last.pt"
resume_state = globals().pop("UGVNET_RESUME_STATE", None)

if resume_state is None:
    history = {
        "train": [],
        "validation": [],
        "epoch_numbers": [],
        "adaptations": [],
    }
    best_validation_f1 = -1.0
    completed_epochs = 0
else:
    history = resume_state.get("history", {"train": [], "validation": []})
    history.setdefault(
        "epoch_numbers",
        history.pop("accepted_attempts", list(range(1, len(history["train"]) + 1))),
    )
    history.setdefault("adaptations", [])
    history.pop("rejected", None)
    completed_epochs = int(resume_state.get("epoch", len(history["train"])))
    best_validation_f1 = max(
        (row["macro_f1"] for row in history["validation"]),
        default=-1.0,
    )
    print(f"Continuing after epoch {completed_epochs}.")

epochs_without_improvement = int(
    resume_state.get("epochs_without_improvement", 0)
    if resume_state is not None
    else 0
)
high_gap_streak = int(
    resume_state.get("high_gap_streak", 0)
    if resume_state is not None
    else 0
)
training_started = time.time()

while completed_epochs < EPOCHS:
    proposed_epoch = completed_epochs + 1
    backbones_trainable = any(
        parameter.requires_grad
        for parameter in model.efficientnet_features.parameters()
    )
    if not backbones_trainable and proposed_epoch > freeze_epochs:
        model.set_backbones_trainable(True)
        backbones_trainable = True
        print("Backbones unfrozen: beginning full fine-tuning.")

    epoch_started = time.time()
    print(
        f"\nEpoch {proposed_epoch:03d}/{EPOCHS}"
    )
    train_metrics = run_epoch(
        model,
        LOADERS["train"],
        training=True,
        split_name="train",
        gradient_histogram_step=(
            proposed_epoch
            if TENSORBOARD_LOG_GRADIENT_HISTOGRAMS
            and proposed_epoch % TENSORBOARD_HISTOGRAM_INTERVAL == 0
            else None
        ),
        profile_batches=(
            TENSORBOARD_PROFILE_BATCHES
            if proposed_epoch == 1
            else 0
        ),
    )
    embedding_due = (
        TENSORBOARD_ENABLED
        and TENSORBOARD_LOG_EMBEDDINGS
        and TENSORBOARD_EMBEDDING_INTERVAL > 0
        and (
            proposed_epoch == 1
            or proposed_epoch % TENSORBOARD_EMBEDDING_INTERVAL == 0
        )
    )
    activation_due = (
        TENSORBOARD_ENABLED
        and TENSORBOARD_LOG_ACTIVATIONS
        and TENSORBOARD_ACTIVATION_INTERVAL > 0
        and (
            proposed_epoch == 1
            or proposed_epoch % TENSORBOARD_ACTIVATION_INTERVAL == 0
        )
    )
    validation_metrics = run_epoch(
        model,
        LOADERS["validation"],
        training=False,
        split_name="validation",
        collect_tensorboard=TENSORBOARD_ENABLED,
        collect_outputs=embedding_due,
        collect_curve_metrics=(
            TENSORBOARD_ENABLED and TENSORBOARD_LOG_CLASS_METRICS
        ),
        activation_histogram_step=(proposed_epoch if activation_due else None),
    )
    validation_tensorboard_outputs = validation_metrics.pop(
        "_tensorboard_outputs",
        None,
    )

    accuracy_gap = train_metrics["accuracy"] - validation_metrics["accuracy"]
    gap_is_high = (
        proposed_epoch >= GAP_ADAPTATION_START_EPOCH
        and accuracy_gap > GENERALIZATION_GAP_THRESHOLD
    )
    high_gap_streak = high_gap_streak + 1 if gap_is_high else 0
    adapt_regularization = high_gap_streak >= GAP_ADAPTATION_PATIENCE

    completed_epochs = proposed_epoch
    scheduler.step()
    history["train"].append(train_metrics)
    history["validation"].append(validation_metrics)
    history["epoch_numbers"].append(completed_epochs)
    adaptation = None
    if adapt_regularization:
        previous_dropout = current_dropout
        previous_label_smoothing = current_label_smoothing
        previous_focal_gamma = current_focal_gamma
        previous_learning_rates = [
            parameter_group["lr"] for parameter_group in optimizer.param_groups
        ]
        current_dropout = min(
            MAX_ADAPTIVE_DROPOUT,
            current_dropout + GAP_ADAPTATION_DROPOUT_STEP,
        )
        current_label_smoothing = min(
            MAX_ADAPTIVE_LABEL_SMOOTHING,
            current_label_smoothing + GAP_ADAPTATION_LABEL_SMOOTHING_STEP,
        )
        if use_focal_loss:
            current_focal_gamma = max(
                MIN_ADAPTIVE_FOCAL_GAMMA,
                current_focal_gamma - GAP_ADAPTATION_FOCAL_GAMMA_STEP,
            )
        set_model_dropout(model, current_dropout)
        criterion = build_criterion(
            current_label_smoothing,
            current_focal_gamma,
        )
        for parameter_group in optimizer.param_groups:
            parameter_group["lr"] *= GAP_ADAPTATION_LR_FACTOR
        adaptation = {
            "epoch": completed_epochs,
            "accuracy_gap": accuracy_gap,
            "dropout_before": previous_dropout,
            "dropout_after": current_dropout,
            "label_smoothing_before": previous_label_smoothing,
            "label_smoothing_after": current_label_smoothing,
            "focal_gamma_before": previous_focal_gamma,
            "focal_gamma_after": current_focal_gamma,
            "learning_rates_before": previous_learning_rates,
            "learning_rates_after": [
                parameter_group["lr"] for parameter_group in optimizer.param_groups
            ],
        }
        history["adaptations"].append(adaptation)
        high_gap_streak = 0
        print(
            f"Generalization gap stayed above {GENERALIZATION_GAP_THRESHOLD:.0%} "
            f"for {GAP_ADAPTATION_PATIENCE} epochs. Epoch {completed_epochs} "
            "was kept; next-epoch regularization was adjusted: "
            f"LR x{GAP_ADAPTATION_LR_FACTOR:.2f}, "
            f"dropout={current_dropout:.2f}, "
            f"label smoothing={current_label_smoothing:.2f}"
            + (
                f", focal gamma={current_focal_gamma:.2f}."
                if use_focal_loss
                else "."
            )
        )
    epoch_learning_rates = [
        parameter_group["lr"] for parameter_group in optimizer.param_groups
    ]
    if (
        TENSORBOARD_HISTOGRAM_INTERVAL > 0
        and completed_epochs % TENSORBOARD_HISTOGRAM_INTERVAL == 0
    ):
        tensorboard_log_histograms(model, completed_epochs)
    if (
        TENSORBOARD_OPTIMIZER_INTERVAL > 0
        and completed_epochs % TENSORBOARD_OPTIMIZER_INTERVAL == 0
    ):
        tensorboard_log_optimizer_histograms(
            optimizer,
            model,
            completed_epochs,
        )
    if validation_tensorboard_outputs is not None:
        embedding_targets = validation_tensorboard_outputs["targets"]
        embedding_values = validation_tensorboard_outputs["embeddings"]
        sample_count = min(
            TENSORBOARD_MAX_EMBEDDING_SAMPLES,
            embedding_values.size(0),
        )
        TENSORBOARD_WRITER.add_embedding(
            embedding_values[:sample_count],
            metadata=[
                CLASSES[index]
                for index in embedding_targets[:sample_count].tolist()
            ],
            global_step=completed_epochs,
            tag="Projector/validation_embedding_evolution",
        )

    for split_name, metrics in (
        ("train", train_metrics),
        ("validation", validation_metrics),
    ):
        for metric_name in (
            "loss",
            "accuracy",
            "macro_f1",
            "images_per_second",
        ):
            tensorboard_scalar(
                f"Epoch/{split_name}/{metric_name}",
                metrics[metric_name],
                completed_epochs,
            )
    tensorboard_scalar("Epoch/generalization_gap", accuracy_gap, completed_epochs)
    tensorboard_scalar("Epoch/gap_is_high", gap_is_high, completed_epochs)
    tensorboard_scalar(
        "Epoch/regularization_adapted",
        adaptation is not None,
        completed_epochs,
    )
    if "gradient_norm" in train_metrics:
        tensorboard_scalar(
            "Epoch/train/gradient_norm",
            train_metrics["gradient_norm"],
            completed_epochs,
        )
    for metric_name, value in validation_metrics.get(
        "hybrid_diagnostics",
        {},
    ).items():
        tensorboard_scalar(
            f"Hybrid/{metric_name}",
            value,
            completed_epochs,
        )
    tensorboard_log_class_metrics(
        "Validation/ClassMetrics",
        validation_metrics,
        completed_epochs,
    )
    tensorboard_scalar("Runtime/dropout", current_dropout, completed_epochs)
    tensorboard_scalar(
        "Runtime/label_smoothing",
        current_label_smoothing,
        completed_epochs,
    )
    tensorboard_scalar(
        "Runtime/focal_gamma",
        current_focal_gamma if use_focal_loss else 0.0,
        completed_epochs,
    )
    tensorboard_scalar(
        "Run/regularization_adaptations",
        len(history["adaptations"]),
        completed_epochs,
    )
    for group_index, learning_rate in enumerate(epoch_learning_rates):
        tensorboard_scalar(
            f"Learning_rate/group_{group_index}",
            learning_rate,
            completed_epochs,
        )
    tensorboard_log_system_metrics(completed_epochs)

    backbones_trainable = any(
        parameter.requires_grad
        for parameter in model.efficientnet_features.parameters()
    )
    checkpoint = {
        "epoch": completed_epochs,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "classes": CLASSES,
        "validation_metrics": validation_metrics,
        "history": history,
        "best_validation_f1": best_validation_f1,
        "backbones_trainable": backbones_trainable,
        "runtime_dropout": current_dropout,
        "runtime_label_smoothing": current_label_smoothing,
        "runtime_focal_gamma": current_focal_gamma,
        "high_gap_streak": high_gap_streak,
        "model_config": {
            "num_classes": NUM_CLASSES,
            "image_size": IMAGE_SIZE,
            "fusion_channels": FUSION_CHANNELS,
            "attention_heads": ATTENTION_HEADS,
            "fusion_depth": FUSION_DEPTH,
            "dropout": DROPOUT,
        },
    }
    improved = validation_metrics["macro_f1"] > best_validation_f1
    if improved:
        best_validation_f1 = validation_metrics["macro_f1"]
        epochs_without_improvement = 0
        best_checkpoint = {
            "epoch": completed_epochs,
                "model": model.state_dict(),
            "classes": CLASSES,
            "validation_metrics": validation_metrics,
            "accuracy_gap": accuracy_gap,
            "model_config": checkpoint["model_config"],
        }
        torch.save(best_checkpoint, best_checkpoint_path)
    else:
        epochs_without_improvement += 1
    checkpoint["best_validation_f1"] = best_validation_f1
    checkpoint["epochs_without_improvement"] = epochs_without_improvement
    checkpoint["loss_function"] = RESOLVED_LOSS_FUNCTION
    numpy_rng_state = np.random.get_state()
    checkpoint["rng_state"] = {
        "python": random.getstate(),
        "numpy": (
            numpy_rng_state[0],
            numpy_rng_state[1].tolist(),
            numpy_rng_state[2],
            numpy_rng_state[3],
            numpy_rng_state[4],
        ),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else [],
    }
    torch.save(checkpoint, last_checkpoint_path)
    tensorboard_scalar("Epoch/is_best", improved, completed_epochs)
    tensorboard_text(
        "Artifacts/checkpoints",
        (
            "| Artifact | Path | Status |  \n"
            "|---|---|---|  \n"
            f"| Best model | `{best_checkpoint_path}` | "
            f"{'updated' if improved else 'unchanged'} |  \n"
            f"| Last checkpoint | `{last_checkpoint_path}` | updated |"
        ),
        completed_epochs,
    )

    elapsed = time.time() - epoch_started
    print(
        f"Epoch {completed_epochs:03d}/{EPOCHS} ({elapsed:.1f}s) | "
        f"gap={accuracy_gap:.2%} | "
        f"train: {format_metrics(train_metrics)} | "
        f"validation: {format_metrics(validation_metrics)} "
        f"{'★ best' if improved else ''}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"Early stopping at epoch {completed_epochs}; "
            f"no macro-F1 improvement for {PATIENCE} epochs."
        )
        break

with (RESULTS_DIR / "history.json").open("w") as file:
    json.dump(history, file, indent=2)
TRAINING_DURATION_SECONDS = time.time() - training_started
print(f"Training finished in {TRAINING_DURATION_SECONDS / 60:.1f} minutes.")
print(
    f"Completed epochs: {completed_epochs} | "
    f"Regularization adaptations: {len(history['adaptations'])}"
)
if not history["train"]:
    raise RuntimeError("Training ended without a completed epoch.")
if (
    TENSORBOARD_LOG_OPTIMIZER_STATE
    and TENSORBOARD_OPTIMIZER_INTERVAL > 0
    and completed_epochs % TENSORBOARD_OPTIMIZER_INTERVAL != 0
):
    tensorboard_log_optimizer_histograms(optimizer, model, completed_epochs)

BEST_TRAINING_EPOCH = max(
    range(1, len(history["train"]) + 1),
    key=lambda epoch: history["train"][epoch - 1]["accuracy"],
)
BEST_TRAINING_ACCURACY = history["train"][BEST_TRAINING_EPOCH - 1]["accuracy"]
BEST_VALIDATION_EPOCH = max(
    range(1, len(history["validation"]) + 1),
    key=lambda epoch: history["validation"][epoch - 1]["accuracy"],
)
BEST_VALIDATION_ACCURACY = history["validation"][
    BEST_VALIDATION_EPOCH - 1
]["accuracy"]

print(
    f"Best Training Accuracy: {BEST_TRAINING_ACCURACY:.4f} | "
    f"Best Training Accuracy in %: {BEST_TRAINING_ACCURACY:.2%} | "
    f"Epoch: {BEST_TRAINING_EPOCH}"
)
print(
    f"Best Validation Accuracy: {BEST_VALIDATION_ACCURACY:.4f} | "
    f"Best Validation Accuracy in %: {BEST_VALIDATION_ACCURACY:.2%} | "
    f"Epoch: {BEST_VALIDATION_EPOCH}"
)
print(f"Best validation macro-F1: {best_validation_f1:.4f}")
tensorboard_scalar(
    "Best/training_accuracy",
    BEST_TRAINING_ACCURACY,
    BEST_TRAINING_EPOCH,
)
tensorboard_scalar(
    "Best/validation_accuracy",
    BEST_VALIDATION_ACCURACY,
    BEST_VALIDATION_EPOCH,
)
tensorboard_scalar("Run/completed_epochs", completed_epochs, completed_epochs)
tensorboard_scalar(
    "Run/regularization_adaptations",
    len(history["adaptations"]),
    completed_epochs,
)
if TENSORBOARD_WRITER is not None and not TENSORBOARD_WRITER_CLOSED:
    TENSORBOARD_WRITER.flush()

## 18. Training curves

**This cell covers:** Train-versus-validation loss, accuracy, and macro-F1 across every completed epoch. Generalization-gap adaptations remain recorded inside `history.json` and TensorBoard.


In [ ]:
# 18. Training plots
accepted_epoch_numbers = np.arange(1, len(history["train"]) + 1)
_curve_colors = {"train": "#3b82f6", "validation": "#22c55e"}
_curve_fill   = {"train": "#93c5fd", "validation": "#86efac"}

figure, axes = plt.subplots(1, 3, figsize=(18, 5.5), facecolor="#fafbff")
for axis, metric, title in zip(
    axes,
    ("loss", "accuracy", "macro_f1"),
    ("Loss", "Accuracy", "Macro-F1"),
):
    train_values = [item[metric] for item in history["train"]]
    val_values = [item[metric] for item in history["validation"]]
    axis.plot(
        accepted_epoch_numbers, train_values,
        color=_curve_colors["train"], linewidth=1.8, label="Train",
    )
    axis.plot(
        accepted_epoch_numbers, val_values,
        color=_curve_colors["validation"], linewidth=1.8, label="Validation",
    )
    axis.fill_between(
        accepted_epoch_numbers, train_values,
        alpha=0.08, color=_curve_fill["train"],
    )
    axis.fill_between(
        accepted_epoch_numbers, val_values,
        alpha=0.08, color=_curve_fill["validation"],
    )
    if metric != "loss":
        best_idx = int(np.argmax(val_values))
    else:
        best_idx = int(np.argmin(val_values))
    axis.axvline(
        x=best_idx + 1, color="#e2e8f0",
        linestyle="--", linewidth=1, alpha=0.7,
    )
    axis.scatter(
        [best_idx + 1], [val_values[best_idx]],
        color=_curve_colors["validation"],
        s=40, zorder=5, edgecolor="white", linewidth=1.5,
    )
    axis.set_facecolor("#fafbff")
    axis.set_title(
        title, fontsize=14, fontfamily="Inter",
        fontweight="700", pad=10,
    )
    axis.set_xlabel("Epoch", fontsize=11, fontfamily="Inter", fontweight="500")
    axis.set_ylabel(title, fontsize=11, fontfamily="Inter", fontweight="500")
    axis.legend(
        frameon=True, fancybox=True, framealpha=0.85,
        edgecolor="#cbd5e1", fontsize=9,
        prop={"family": "Inter", "weight": "500"},
    )
    axis.grid(axis="y", linestyle="--", alpha=0.4, color="#94a3b8")
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_color("#cbd5e1")
    axis.spines["bottom"].set_color("#cbd5e1")
    axis.tick_params(labelsize=9)

figure.suptitle(
    "UGVNet Training Curves",
    fontsize=16, fontfamily="Inter", fontweight="800", y=1.02,
)
figure.tight_layout()
figure.savefig(
    RESULTS_DIR / "training_curves.png",
    dpi=200, bbox_inches="tight", facecolor="#fafbff",
)
plt.show()


## 19. Evaluate the test set

**This cell covers:** Best-checkpoint evaluation, complete per-class metrics, calibration, numbered confusion matrix, class-wise error ranking, top true/predicted confusion pairs, prediction and fusion images, fused-feature Grad-CAM, PR/ROC curves, final embeddings, HParams, and TensorBoard artifact summaries.

The error tables use the untouched test set only after model selection and are saved as `test_class_error_analysis.csv` and `test_confusion_pairs.csv`.


In [ ]:
# 19. Test evaluation
best_checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_checkpoint["model"])
test_metrics = run_epoch(
    model,
    LOADERS["test"],
    training=False,
    split_name="test",
    collect_tensorboard=TENSORBOARD_ENABLED,
    collect_outputs=TENSORBOARD_ENABLED,
    collect_curve_metrics=(
        TENSORBOARD_ENABLED and TENSORBOARD_LOG_CLASS_METRICS
    ),
    max_prediction_images=TENSORBOARD_MAX_PREDICTION_IMAGES,
)
test_tensorboard_outputs = test_metrics.pop("_tensorboard_outputs", None)
BEST_TEST_ACCURACY = test_metrics["accuracy"]

for metric_name in ("loss", "accuracy", "macro_f1", "images_per_second"):
    tensorboard_scalar(
        f"Test/{metric_name}",
        test_metrics[metric_name],
        best_checkpoint["epoch"],
    )
for class_index, (class_name, score) in enumerate(
    zip(CLASSES, test_metrics["class_f1"])
):
    safe_class_name = "".join(
        character if character.isalnum() else "_"
        for character in class_name
    ).strip("_")
    tensorboard_scalar(
        f"Test/class_f1/{class_index:02d}_{safe_class_name}",
        score,
        best_checkpoint["epoch"],
    )
tensorboard_scalar(
    "Best/test_accuracy",
    BEST_TEST_ACCURACY,
    best_checkpoint["epoch"],
)
tensorboard_log_class_metrics(
    "Test/ClassMetrics",
    test_metrics,
    best_checkpoint["epoch"],
)
tensorboard_text(
    "Final/accuracy_summary",
    (
        f"Best Training Accuracy in %: {BEST_TRAINING_ACCURACY:.2%}  \n"
        f"Best Validation Accuracy in %: {BEST_VALIDATION_ACCURACY:.2%}  \n"
        f"Best Test Accuracy in %: {BEST_TEST_ACCURACY:.2%}"
    ),
    best_checkpoint["epoch"],
)
if TENSORBOARD_WRITER is not None and not TENSORBOARD_WRITER_CLOSED:
    TENSORBOARD_WRITER.flush()


if TENSORBOARD_WRITER is not None and test_tensorboard_outputs is not None:
    test_probabilities = test_tensorboard_outputs["probabilities"]
    test_targets = test_tensorboard_outputs["targets"]
    if TENSORBOARD_LOG_PR_CURVES:
        for class_index, class_name in enumerate(CLASSES):
            binary_targets = (test_targets == class_index).int()
            TENSORBOARD_WRITER.add_pr_curve(
                f"PR/{class_name}",
                binary_targets,
                test_probabilities[:, class_index],
                global_step=best_checkpoint["epoch"],
            )
    if TENSORBOARD_LOG_EMBEDDINGS:
        test_embeddings = test_tensorboard_outputs["embeddings"]
        sample_images = test_tensorboard_outputs.get("sample_images")
        if sample_images is not None:
            sample_count = sample_images.size(0)
            test_embeddings = test_embeddings[:sample_count]
            test_targets_emb = test_targets[:sample_count]
        else:
            test_targets_emb = test_targets

        embedding_metadata = [CLASSES[index] for index in test_targets_emb.tolist()]
        TENSORBOARD_WRITER.add_embedding(
            test_embeddings,
            label_img=sample_images,
            metadata=embedding_metadata,
            global_step=best_checkpoint["epoch"],
            tag="test_fused_embeddings",
        )
    if TENSORBOARD_LOG_IMAGES:
        images = test_tensorboard_outputs["sample_images"]
        targets = test_tensorboard_outputs["sample_targets"]
        predictions = test_tensorboard_outputs["sample_predictions"]
        confidences = test_tensorboard_outputs["sample_confidences"]
        if images is not None:
            mean = torch.tensor((0.485, 0.456, 0.406)).view(1, 3, 1, 1)
            std = torch.tensor((0.229, 0.224, 0.225)).view(1, 3, 1, 1)
            display_images = (images * std + mean).clamp(0, 1)
            columns = 4
            rows = math.ceil(display_images.size(0) / columns)
            prediction_figure, prediction_axes = plt.subplots(
                rows,
                columns,
                figsize=(columns * 3.2, rows * 3.2),
            )
            prediction_axes = np.asarray(prediction_axes).reshape(-1)
            for sample_index, sample_axis in enumerate(prediction_axes):
                sample_axis.axis("off")
                if sample_index >= display_images.size(0):
                    continue
                sample_axis.imshow(
                    display_images[sample_index].permute(1, 2, 0).numpy()
                )
                target_name = CLASSES[int(targets[sample_index])]
                prediction_name = CLASSES[int(predictions[sample_index])]
                correct = target_name == prediction_name
                sample_axis.set_title(
                    f"True: {target_name}\nPred: {prediction_name} "
                    f"({float(confidences[sample_index]):.1%})",
                    color="green" if correct else "crimson",
                    fontsize=9,
                )
            prediction_figure.suptitle("UGVNet test predictions")
            prediction_figure.tight_layout()
            TENSORBOARD_WRITER.add_figure(
                "Images/test_predictions",
                prediction_figure,
                global_step=best_checkpoint["epoch"],
                close=True,
            )
        fusion_maps = test_tensorboard_outputs["fusion_weights"]
        if fusion_maps is not None:
            TENSORBOARD_WRITER.add_images(
                "Images/fusion_gate_efficientnet",
                fusion_maps[:, 0:1],
                global_step=best_checkpoint["epoch"],
            )
            TENSORBOARD_WRITER.add_images(
                "Images/fusion_gate_convnext",
                fusion_maps[:, 1:2],
                global_step=best_checkpoint["epoch"],
            )
    if TENSORBOARD_LOG_ROC_CURVES:
        roc_figure, roc_axis = plt.subplots(figsize=(8, 7))
        plotted_curves = 0
        for class_index, class_name in enumerate(CLASSES):
            false_positive_rate, true_positive_rate, auc = binary_roc_curve(
                test_targets == class_index,
                test_probabilities[:, class_index],
            )
            if auc is None:
                continue
            roc_axis.plot(
                false_positive_rate,
                true_positive_rate,
                linewidth=1.8,
                label=f"{class_name} (AUC={auc:.3f})",
            )
            plotted_curves += 1
        if plotted_curves:
            roc_axis.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
            roc_axis.set(
                xlabel="False positive rate",
                ylabel="True positive rate",
                title="UGVNet one-vs-rest ROC curves",
                xlim=(0, 1),
                ylim=(0, 1),
            )
            roc_axis.legend(fontsize=8, loc="lower right")
            roc_figure.tight_layout()
            TENSORBOARD_WRITER.add_figure(
                "Curves/test_roc",
                roc_figure,
                global_step=best_checkpoint["epoch"],
                close=True,
            )
        else:
            plt.close(roc_figure)

    if TENSORBOARD_LOG_CALIBRATION:
        bin_confidence, bin_accuracy, bin_fraction = reliability_curve(
            test_targets,
            test_probabilities,
        )
        calibration_figure, calibration_axis = plt.subplots(figsize=(7, 6))
        calibration_axis.plot([0, 1], [0, 1], "--", color="gray", label="Ideal")
        if bin_confidence:
            sizes = [max(35, 900 * fraction) for fraction in bin_fraction]
            calibration_axis.scatter(
                bin_confidence,
                bin_accuracy,
                s=sizes,
                alpha=0.8,
                color="#4f7cff",
                label="UGVNet",
            )
            calibration_axis.plot(bin_confidence, bin_accuracy, color="#4f7cff")
        calibration_axis.set(
            xlabel="Mean confidence",
            ylabel="Observed accuracy",
            title=(
                "Reliability diagram "
                f"(ECE={test_metrics['expected_calibration_error']:.3f})"
            ),
            xlim=(0, 1),
            ylim=(0, 1),
        )
        calibration_axis.legend()
        calibration_figure.tight_layout()
        TENSORBOARD_WRITER.add_figure(
            "Curves/test_calibration",
            calibration_figure,
            global_step=best_checkpoint["epoch"],
            close=True,
        )

    if TENSORBOARD_LOG_GRADCAM:
        gradcam_images = test_tensorboard_outputs.get("sample_images")
        gradcam_targets = test_tensorboard_outputs.get("sample_targets")
        if gradcam_images is not None:
            gradcam_count = min(
                TENSORBOARD_MAX_GRADCAM_IMAGES,
                gradcam_images.size(0),
            )
            gradcam_input = gradcam_images[:gradcam_count].to(DEVICE)
            if CHANNELS_LAST_ENABLED:
                gradcam_input = gradcam_input.to(memory_format=torch.channels_last)
            model.eval()
            with torch.no_grad():
                gradcam_features = model.forward_features(gradcam_input)
            gradcam_features = gradcam_features.detach().requires_grad_(True)
            gradcam_logits = model.forward_head(gradcam_features)
            gradcam_predictions = gradcam_logits.argmax(1)
            selected_scores = gradcam_logits.gather(
                1,
                gradcam_predictions.unsqueeze(1),
            ).sum()
            gradcam_gradients = torch.autograd.grad(
                selected_scores,
                gradcam_features,
            )[0]
            gradcam_weights = gradcam_gradients.mean(dim=(2, 3), keepdim=True)
            gradcam_maps = torch.relu(
                (gradcam_weights * gradcam_features).sum(dim=1, keepdim=True)
            )
            gradcam_maps = F.interpolate(
                gradcam_maps,
                size=gradcam_input.shape[-2:],
                mode="bilinear",
                align_corners=False,
            ).cpu()
            flat_maps = gradcam_maps.flatten(1)
            map_minimum = flat_maps.min(1).values.view(-1, 1, 1, 1)
            map_maximum = flat_maps.max(1).values.view(-1, 1, 1, 1)
            gradcam_maps = (gradcam_maps - map_minimum) / (
                map_maximum - map_minimum
            ).clamp_min(1e-8)
            mean = torch.tensor((0.485, 0.456, 0.406)).view(1, 3, 1, 1)
            std = torch.tensor((0.229, 0.224, 0.225)).view(1, 3, 1, 1)
            gradcam_display = (gradcam_images[:gradcam_count] * std + mean).clamp(0, 1)
            gradcam_figure, gradcam_axes = plt.subplots(
                gradcam_count,
                2,
                figsize=(7, max(3.5, gradcam_count * 3.1)),
                squeeze=False,
            )
            for sample_index in range(gradcam_count):
                image_array = gradcam_display[sample_index].permute(1, 2, 0).detach().cpu().numpy()
                heatmap = plt.get_cmap("jet")(
                    gradcam_maps[sample_index, 0].detach().cpu().numpy()
                )[:, :, :3]
                overlay = np.clip(0.58 * image_array + 0.42 * heatmap, 0, 1)
                true_name = CLASSES[int(gradcam_targets[sample_index])]
                predicted_name = CLASSES[int(gradcam_predictions[sample_index].cpu())]
                gradcam_axes[sample_index, 0].imshow(image_array)
                gradcam_axes[sample_index, 0].set_title(f"Input: {true_name}")
                gradcam_axes[sample_index, 1].imshow(overlay)
                gradcam_axes[sample_index, 1].set_title(f"Grad-CAM: {predicted_name}")
                gradcam_axes[sample_index, 0].axis("off")
                gradcam_axes[sample_index, 1].axis("off")
            gradcam_figure.suptitle("UGVNet fused-feature Grad-CAM")
            gradcam_figure.tight_layout()
            TENSORBOARD_WRITER.add_figure(
                "Explainability/test_gradcam",
                gradcam_figure,
                global_step=best_checkpoint["epoch"],
                close=True,
            )
            del gradcam_input, gradcam_features, gradcam_gradients, gradcam_logits
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    tensorboard_log_full_model_histograms(model, best_checkpoint["epoch"])
    tensorboard_log_system_metrics(best_checkpoint["epoch"])

    hparam_metrics = {
        "hparam/best_training_accuracy": BEST_TRAINING_ACCURACY,
        "hparam/best_validation_accuracy": BEST_VALIDATION_ACCURACY,
        "hparam/test_accuracy": BEST_TEST_ACCURACY,
        "hparam/test_macro_f1": test_metrics["macro_f1"],
        "hparam/test_macro_roc_auc": (test_metrics.get("macro_roc_auc") or 0.0),
        "hparam/test_calibration_error": test_metrics.get(
            "expected_calibration_error",
            0.0,
        ),
    }
    TENSORBOARD_WRITER.add_hparams(
        run_configuration,
        hparam_metrics,
        run_name="hparams",
    )

print(f"Best validation-selected epoch: {best_checkpoint['epoch']}")
print(
    f"Best Training Accuracy: {BEST_TRAINING_ACCURACY:.4f} | "
    f"Best Training Accuracy in %: {BEST_TRAINING_ACCURACY:.2%} | "
    f"Epoch: {BEST_TRAINING_EPOCH}"
)
print(
    f"Best Validation Accuracy: {BEST_VALIDATION_ACCURACY:.4f} | "
    f"Best Validation Accuracy in %: {BEST_VALIDATION_ACCURACY:.2%} | "
    f"Epoch: {BEST_VALIDATION_EPOCH}"
)
print(
    f"Best Test Accuracy: {BEST_TEST_ACCURACY:.4f} | "
    f"Best Test Accuracy in %: {BEST_TEST_ACCURACY:.2%}"
)
print(f"Test: {format_metrics(test_metrics)}")

show_accuracy_summary([
    (
        "Best Training Accuracy in %",
        BEST_TRAINING_ACCURACY,
        f"epoch {BEST_TRAINING_EPOCH}",
    ),
    (
        "Best Validation Accuracy in %",
        BEST_VALIDATION_ACCURACY,
        f"epoch {BEST_VALIDATION_EPOCH}",
    ),
    (
        "Best Test Accuracy in %",
        BEST_TEST_ACCURACY,
        "validation-selected model",
    ),
])

print("\nPer-class test F1:")
for class_name, score in zip(CLASSES, test_metrics["class_f1"]):
    print(f"  {class_name:30s}: {score:.4f}")


confusion = np.asarray(test_metrics["confusion"], dtype=np.int64)
class_error_rows = []
confusion_pair_rows = []
for true_index, class_name in enumerate(CLASSES):
    support = int(confusion[true_index].sum())
    correct = int(confusion[true_index, true_index])
    errors = support - correct
    error_rate = errors / support if support else 0.0
    off_diagonal = confusion[true_index].copy()
    off_diagonal[true_index] = 0
    confused_index = int(off_diagonal.argmax()) if off_diagonal.size else true_index
    confused_count = int(off_diagonal[confused_index]) if off_diagonal.size else 0
    class_error_rows.append({
        "class": class_name,
        "train_images": int(counts[true_index]),
        "test_images": support,
        "correct": correct,
        "errors": errors,
        "error_rate": error_rate,
        "error_rate_percent": error_rate * 100,
        "most_confused_with": (
            CLASSES[confused_index] if confused_count else "None"
        ),
        "most_confused_count": confused_count,
        "most_confused_rate_percent": (
            100 * confused_count / support if support else 0.0
        ),
    })
    for predicted_index, pair_count in enumerate(confusion[true_index]):
        if predicted_index == true_index or pair_count == 0:
            continue
        confusion_pair_rows.append({
            "true_class": class_name,
            "predicted_class": CLASSES[predicted_index],
            "image_count": int(pair_count),
            "percent_of_true_class": 100 * int(pair_count) / support if support else 0.0,
        })

TEST_CLASS_ERROR_TABLE = pd.DataFrame(class_error_rows).sort_values(
    ["error_rate", "errors"],
    ascending=False,
    ignore_index=True,
)
TEST_CONFUSION_PAIRS_TABLE = pd.DataFrame(confusion_pair_rows)
if not TEST_CONFUSION_PAIRS_TABLE.empty:
    TEST_CONFUSION_PAIRS_TABLE = TEST_CONFUSION_PAIRS_TABLE.sort_values(
        ["image_count", "percent_of_true_class"],
        ascending=False,
        ignore_index=True,
    )
else:
    TEST_CONFUSION_PAIRS_TABLE = pd.DataFrame(columns=[
        "true_class",
        "predicted_class",
        "image_count",
        "percent_of_true_class",
    ])

show_dataframe(
    TEST_CLASS_ERROR_TABLE.rename(
        columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
    ),
    "Class-wise test error analysis",
)
show_dataframe(
    TEST_CONFUSION_PAIRS_TABLE.head(15).rename(
        columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
    ),
    "Most frequent test-set confusions",
)
TEST_CLASS_ERROR_TABLE.rename(
    columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
).to_csv(
    RESULTS_DIR / "test_class_error_analysis.csv",
    index=False,
)
TEST_CONFUSION_PAIRS_TABLE.rename(
    columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
).to_csv(
    RESULTS_DIR / "test_confusion_pairs.csv",
    index=False,
)

for class_index, row in TEST_CLASS_ERROR_TABLE.iterrows():
    safe_name = safe_tensorboard_name(row["class"])
    tensorboard_scalar(
        f"Test/class_error_rate/{class_index:02d}_{safe_name}",
        row["error_rate"],
        best_checkpoint["epoch"],
    )
if not TEST_CONFUSION_PAIRS_TABLE.empty:
    markdown_rows = [
        "| True class | Predicted class | Images | True-class share |",
        "|---|---|---:|---:|",
    ]
    for row in TEST_CONFUSION_PAIRS_TABLE.head(15).itertuples(index=False):
        markdown_rows.append(
            f"| {row.true_class} | {row.predicted_class} | "
            f"{row.image_count} | {row.percent_of_true_class:.2f}% |"
        )
    tensorboard_text(
        "Final/top_confusion_pairs",
        "  \n".join(markdown_rows),
        best_checkpoint["epoch"],
    )

def annotate_confusion_matrix(axis, matrix):
    """Draw count and row-percentage labels over each confusion-matrix cell."""
    matrix = np.asarray(matrix)
    maximum = float(matrix.max()) if matrix.size else 0.0
    contrast_threshold = maximum * 0.5
    row_sums = matrix.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    font_size = max(5.5, min(11.0, 80.0 / max(1, matrix.shape[0])))
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            count = int(matrix[row_index, column_index])
            if count == 0:
                continue
            pct = matrix[row_index, column_index] / row_sums[row_index, 0] * 100
            text_color = "white" if count > contrast_threshold else "#1e293b"
            axis.text(
                column_index, row_index - 0.13, f"{count:,}",
                ha="center", va="center",
                fontsize=font_size, fontweight="600",
                fontfamily="Inter", color=text_color,
            )
            axis.text(
                column_index, row_index + 0.18, f"{pct:.0f}%",
                ha="center", va="center",
                fontsize=font_size - 2, fontweight="400",
                fontfamily="Inter", color=text_color, alpha=0.85,
            )


confusion = np.asarray(test_metrics["confusion"])
figsize = max(7, NUM_CLASSES * 0.65 + 1)
figure, axis = plt.subplots(figsize=(figsize, figsize), facecolor="#fafbff")
axis.set_facecolor("#fafbff")
image = axis.imshow(confusion, cmap="Blues", aspect="equal")
annotate_confusion_matrix(axis, confusion)
if TENSORBOARD_WRITER is not None and TENSORBOARD_LOG_IMAGES:
    TENSORBOARD_WRITER.add_figure(
        "Images/test_confusion_matrix",
        figure,
        global_step=best_checkpoint["epoch"],
        close=False,
    )
axis.set_title(
    "UGVNet Test Confusion Matrix",
    fontsize=15, fontfamily="Inter", fontweight="700", pad=14,
)
axis.set_xlabel("Predicted", fontsize=12, fontfamily="Inter", fontweight="500")
axis.set_ylabel("True", fontsize=12, fontfamily="Inter", fontweight="500")
axis.set_xticks(range(NUM_CLASSES))
axis.set_xticklabels(CLASSES, rotation=40, ha="right", fontsize=10, fontfamily="Inter")
axis.set_yticks(range(NUM_CLASSES))
axis.set_yticklabels(CLASSES, fontsize=10, fontfamily="Inter")
cbar = figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=9)
figure.tight_layout()
figure.savefig(
    RESULTS_DIR / "test_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight",
    facecolor="#fafbff",
)
plt.show()

final_results = {
    "best_epoch": best_checkpoint["epoch"],
    "best_training_accuracy_epoch": BEST_TRAINING_EPOCH,
    "best_training_accuracy": BEST_TRAINING_ACCURACY,
    "best_training_accuracy_percent": BEST_TRAINING_ACCURACY * 100,
    "best_validation_accuracy_epoch": BEST_VALIDATION_EPOCH,
    "best_validation_accuracy": BEST_VALIDATION_ACCURACY,
    "best_validation_accuracy_percent": BEST_VALIDATION_ACCURACY * 100,
    "best_test_accuracy": BEST_TEST_ACCURACY,
    "best_test_accuracy_percent": BEST_TEST_ACCURACY * 100,
    "validation": best_checkpoint["validation_metrics"],
    "test": test_metrics,
    "test_class_error_analysis": TEST_CLASS_ERROR_TABLE.to_dict("records"),
    "top_confusion_pairs": TEST_CONFUSION_PAIRS_TABLE.head(15).to_dict("records"),
    "regularization_adaptations": history.get("adaptations", []),
    "gap_epoch_rejection": False,
    # Backward-compatible percentage fields retained for existing consumers.
    "validation_accuracy_percent": (
        best_checkpoint["validation_metrics"]["accuracy"] * 100
    ),
    "test_accuracy_percent": BEST_TEST_ACCURACY * 100,
    "classes": CLASSES,
    "tensorboard_run_dir": (
        str(TENSORBOARD_RUN_DIR) if TENSORBOARD_RUN_DIR is not None else None
    ),
}
with (RESULTS_DIR / "final_metrics.json").open("w") as file:
    json.dump(final_results, file, indent=2)

## 20. Export styled results and create the training PDF

**This cell covers:** Saving the main run tables as portable CSV files, displaying them with a modern notebook design, and generating a timestamped, multi-page PDF report.

CSV is intentionally kept as plain data so it remains compatible with spreadsheets and analysis tools. Fonts, colors, spacing, and table design are applied to the notebook display and the PDF report.

The report includes the run configuration, complete dataset audit summary, class distribution, training curves, best epoch, decimal and percentage training/validation/test accuracy, complete test metrics, confusion matrix, per-class F1 scores, class-wise error analysis, the most frequent confusion pairs, and artifact locations.

The report manifest and final metrics include the TensorBoard run directory for traceability.


In [ ]:
# 20. Styled CSV exports and PDF report
import textwrap
from datetime import datetime, timezone

from matplotlib.backends.backend_pdf import PdfPages

REPORT_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

history_rows = []
for epoch_index, (train_row, validation_row) in enumerate(
    zip(history["train"], history["validation"]),
    start=1,
):
    history_rows.append({
        "epoch": epoch_index,
        "accuracy_gap": train_row["accuracy"] - validation_row["accuracy"],
        "train_loss": train_row["loss"],
        "validation_loss": validation_row["loss"],
        "train_accuracy": train_row["accuracy"],
        "validation_accuracy": validation_row["accuracy"],
        "train_macro_f1": train_row["macro_f1"],
        "validation_macro_f1": validation_row["macro_f1"],
        "train_images_per_second": train_row.get("images_per_second"),
        "validation_images_per_second": validation_row.get("images_per_second"),
    })
TRAINING_HISTORY_TABLE = pd.DataFrame(history_rows)

test_support = np.asarray(test_metrics["confusion"]).sum(axis=1)
TEST_CLASS_METRICS_TABLE = pd.DataFrame({
    "class": CLASSES,
    "test_support": test_support.astype(int),
    "test_precision": test_metrics["class_precision"],
    "test_recall": test_metrics["class_recall"],
    "test_specificity": test_metrics["class_specificity"],
    "test_f1": test_metrics["class_f1"],
    "test_roc_auc": test_metrics["class_roc_auc"],
})

audit_counts = (
    DATASET_AUDIT["class_counts"]
    if DATASET_AUDIT is not None
    else {
        split: dict(Counter(DATASETS[split].classes[target] for target in DATASETS[split].targets))
        for split in ("train", "validation", "test")
    }
)
DATASET_DISTRIBUTION_TABLE = pd.DataFrame([
    {
        "class": class_name,
        "train": audit_counts["train"].get(class_name, 0),
        "validation": audit_counts["validation"].get(class_name, 0),
        "test": audit_counts["test"].get(class_name, 0),
    }
    for class_name in CLASSES
])
DATASET_DISTRIBUTION_TABLE["total"] = DATASET_DISTRIBUTION_TABLE[
    ["train", "validation", "test"]
].sum(axis=1)

valid_images = (
    DATASET_AUDIT["summary"]["valid_images"]
    if DATASET_AUDIT is not None
    else sum(len(dataset) for dataset in DATASETS.values())
)
critical_issues = (
    DATASET_AUDIT["summary"]["critical_issue_count"]
    if DATASET_AUDIT is not None
    else "Audit disabled"
)
RUN_SUMMARY_TABLE = pd.DataFrame([
    {"item": "Model", "value": "UGVNet - EfficientNetV2-S + ConvNeXt-Tiny"},
    {"item": "Runtime profile", "value": PLATFORM},
    {"item": "Training mode", "value": RESOLVED_MODE},
    {
        "item": "TensorBoard run",
        "value": (
            str(TENSORBOARD_RUN_DIR)
            if TENSORBOARD_RUN_DIR is not None
            else "Disabled"
        ),
    },
    {"item": "Classes", "value": NUM_CLASSES},
    {"item": "Valid images", "value": valid_images},
    {"item": "Dataset critical issues", "value": critical_issues},
    {"item": "Completed epochs", "value": len(history["train"])},
    {
        "item": "Regularization adaptations",
        "value": len(history["adaptations"]),
    },
    {
        "item": "Gap adaptation threshold",
        "value": f"{GENERALIZATION_GAP_THRESHOLD:.2%}",
    },
    {
        "item": "Best validation macro-F1 epoch",
        "value": best_checkpoint["epoch"],
    },
    {
        "item": "Best Training Accuracy epoch",
        "value": BEST_TRAINING_EPOCH,
    },
    {
        "item": "Best Training Accuracy",
        "value": BEST_TRAINING_ACCURACY,
    },
    {
        "item": "Best Training Accuracy in %",
        "value": f"{BEST_TRAINING_ACCURACY:.2%}",
    },
    {
        "item": "Best Validation Accuracy epoch",
        "value": BEST_VALIDATION_EPOCH,
    },
    {
        "item": "Best Validation Accuracy",
        "value": BEST_VALIDATION_ACCURACY,
    },
    {
        "item": "Best Validation Accuracy in %",
        "value": f"{BEST_VALIDATION_ACCURACY:.2%}",
    },
    {
        "item": "Best validation macro-F1",
        "value": best_checkpoint["validation_metrics"]["macro_f1"],
    },
    {"item": "Test loss", "value": test_metrics["loss"]},
    {"item": "Best Test Accuracy", "value": BEST_TEST_ACCURACY},
    {
        "item": "Best Test Accuracy in %",
        "value": f"{BEST_TEST_ACCURACY:.2%}",
    },
    {"item": "Test macro-F1", "value": test_metrics["macro_f1"]},
    {"item": "Training time (minutes)", "value": TRAINING_DURATION_SECONDS / 60},
    {"item": "Total parameters", "value": total_parameters},
    {"item": "Initially trainable parameters", "value": trainable_parameters},
])

CONFUSION_MATRIX_TABLE = pd.DataFrame(
    np.asarray(test_metrics["confusion"]),
    index=CLASSES,
    columns=CLASSES,
)
CONFUSION_MATRIX_TABLE.index.name = "true_class"

CSV_ARTIFACTS = {
    "run_summary": RESULTS_DIR / "run_summary.csv",
    "training_history": RESULTS_DIR / "training_history.csv",
    "dataset_class_distribution": RESULTS_DIR / "dataset_class_distribution.csv",
    "test_class_metrics": RESULTS_DIR / "test_class_metrics.csv",
    "test_class_error_analysis": RESULTS_DIR / "test_class_error_analysis.csv",
    "test_confusion_pairs": RESULTS_DIR / "test_confusion_pairs.csv",
    "test_confusion_matrix": RESULTS_DIR / "test_confusion_matrix.csv",
}
RUN_SUMMARY_TABLE.rename(
    columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
).to_csv(CSV_ARTIFACTS["run_summary"], index=False)
TRAINING_HISTORY_TABLE.rename(
    columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
).to_csv(CSV_ARTIFACTS["training_history"], index=False)
DATASET_DISTRIBUTION_TABLE.to_csv(
    CSV_ARTIFACTS["dataset_class_distribution"],
    index=False,
)
TEST_CLASS_METRICS_TABLE.rename(
    columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
).to_csv(CSV_ARTIFACTS["test_class_metrics"], index=False)
TEST_CLASS_ERROR_TABLE.rename(
    columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
).to_csv(
    CSV_ARTIFACTS["test_class_error_analysis"],
    index=False,
)
TEST_CONFUSION_PAIRS_TABLE.rename(
    columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
).to_csv(
    CSV_ARTIFACTS["test_confusion_pairs"],
    index=False,
)
CONFUSION_MATRIX_TABLE.rename_axis("True Class").to_csv(CSV_ARTIFACTS["test_confusion_matrix"])

show_dataframe(
    RUN_SUMMARY_TABLE.rename(
        columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
    ),
    "UGVNet run summary",
)
show_dataframe(
    TRAINING_HISTORY_TABLE.tail(10).rename(
        columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
    ),
    "Latest training epochs",
)
show_dataframe(
    TEST_CLASS_METRICS_TABLE.rename(
        columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
    ),
    "Per-class test performance",
)
show_dataframe(
    TEST_CLASS_ERROR_TABLE.rename(
        columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
    ),
    "Class-wise test errors",
)
show_dataframe(
    TEST_CONFUSION_PAIRS_TABLE.head(15).rename(
        columns=lambda c: c.replace("_", " ").title() if not c.isupper() else c,
    ),
    "Top confusion pairs",
)


REPORT_COLORS = {
    "navy": "#1e3a5f",
    "blue": "#6366f1",
    "cyan": "#818cf8",
    "ink": "#1e293b",
    "muted": "#64748b",
    "paper": "#fafbff",
    "line": "#e2e8f0",
    "green": "#22c55e",
    "header_bg": "#e0ecff",
    "header_text": "#1e3a5f",
    "alt_row": "#f0f4ff",
}
REPORT_FONT = "Inter"
REPORT_PAGE_SIZE = (8.27, 11.69)


def add_report_frame(figure, title, subtitle="UGVNet training report"):
    figure.patch.set_facecolor(REPORT_COLORS["paper"])
    figure.text(
        0.065,
        0.955,
        title,
        color=REPORT_COLORS["ink"],
        fontsize=20,
        fontweight="bold",
        fontfamily=REPORT_FONT,
        va="top",
    )
    figure.text(
        0.065,
        0.922,
        subtitle,
        color=REPORT_COLORS["muted"],
        fontsize=9,
        fontfamily=REPORT_FONT,
        va="top",
    )
    figure.add_artist(plt.Line2D(
        [0.065, 0.935],
        [0.90, 0.90],
        transform=figure.transFigure,
        color=REPORT_COLORS["blue"],
        linewidth=2.4,
    ))
    figure.text(
        0.065,
        0.028,
        f"Generated {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} | {PLATFORM}",
        color=REPORT_COLORS["muted"],
        fontsize=7.5,
        fontfamily=REPORT_FONT,
    )


def format_report_value(value):
    if isinstance(value, (float, np.floating)):
        if np.isfinite(value) and float(value).is_integer():
            return f"{int(value):,}"
        return f"{value:,.4f}"
    if isinstance(value, (int, np.integer)):
        return f"{value:,}"
    return str(value)


_REPORT_PAGE_NUMBER = 0


def save_report_page(pdf, figure):
    global _REPORT_PAGE_NUMBER
    _REPORT_PAGE_NUMBER += 1
    figure.text(
        0.935,
        0.028,
        f"Page {_REPORT_PAGE_NUMBER}",
        ha="right",
        color=REPORT_COLORS["muted"],
        fontsize=7.5,
        fontfamily=REPORT_FONT,
    )
    pdf.savefig(figure, facecolor=figure.get_facecolor())
    plt.close(figure)


def add_dataframe_pages(pdf, dataframe, title, rows_per_page=24):
    pages_written = 0
    total_chunks = max(1, math.ceil(len(dataframe) / rows_per_page))
    for chunk_index in range(total_chunks):
        start = chunk_index * rows_per_page
        chunk = dataframe.iloc[start:start + rows_per_page].copy()
        figure = plt.figure(figsize=REPORT_PAGE_SIZE)
        page_title = title
        if total_chunks > 1:
            page_title += f" ({chunk_index + 1}/{total_chunks})"
        add_report_frame(figure, page_title)
        axis = figure.add_axes([0.06, 0.08, 0.88, 0.79])
        axis.axis("off")

        wrapped_rows = []
        for row in chunk.itertuples(index=False, name=None):
            wrapped_rows.append([
                "\n".join(textwrap.wrap(format_report_value(value), width=28)) or " "
                for value in row
            ])
        table = axis.table(
            cellText=wrapped_rows,
            colLabels=[str(column).replace("_", " ").title() for column in chunk.columns],
            cellLoc="left",
            colLoc="left",
            loc="upper center",
            bbox=[0.0, 0.02, 1.0, 0.96],
        )
        table.auto_set_font_size(False)
        table.set_fontsize(7.6 if len(chunk.columns) > 4 else 8.4)
        for (row_index, _), cell in table.get_celld().items():
            cell.set_edgecolor(REPORT_COLORS["line"])
            cell.set_linewidth(0.55)
            cell.get_text().set_fontfamily(REPORT_FONT)
            if row_index == 0:
                cell.set_facecolor(REPORT_COLORS.get("header_bg", "#e0ecff"))
                cell.get_text().set_color(REPORT_COLORS.get("header_text", "#1e3a5f"))
                cell.get_text().set_fontweight("bold")
            else:
                cell.set_facecolor(
                    "white" if row_index % 2
                    else REPORT_COLORS.get("alt_row", "#f0f4ff")
                )
                cell.get_text().set_color(REPORT_COLORS["ink"])
        save_report_page(pdf, figure)
        pages_written += 1
    return pages_written


def build_training_pdf(output_path):
    global _REPORT_PAGE_NUMBER
    _REPORT_PAGE_NUMBER = 0
    page_count = 0
    with PdfPages(output_path) as pdf:
        metadata = pdf.infodict()
        metadata["Title"] = "UGVNet Complete Training Report"
        metadata["Author"] = "UGVNet"
        metadata["Subject"] = "Dataset audit, training history, and final evaluation"
        metadata["Keywords"] = "UGVNet, EfficientNetV2-S, ConvNeXt-Tiny, training report"

        cover = plt.figure(figsize=REPORT_PAGE_SIZE)
        cover.patch.set_facecolor(REPORT_COLORS["paper"])
        cover.add_artist(plt.Rectangle(
            (0, 0.73),
            1,
            0.27,
            transform=cover.transFigure,
            color=REPORT_COLORS["navy"],
            zorder=-1,
        ))
        cover.text(
            0.07,
            0.90,
            "UGVNet",
            color="white",
            fontsize=32,
            fontweight="bold",
            fontfamily=REPORT_FONT,
        )
        cover.text(
            0.07,
            0.842,
            "Unified Global Vision Network",
            color="#d9e5ff",
            fontsize=16,
            fontfamily=REPORT_FONT,
        )
        cover.text(
            0.07,
            0.785,
            "Complete training and evaluation report",
            color="white",
            fontsize=11,
            fontfamily=REPORT_FONT,
        )
        cover.text(
            0.07,
            0.685,
            "EfficientNetV2-S + ConvNeXt-Tiny",
            color=REPORT_COLORS["blue"],
            fontsize=17,
            fontweight="bold",
            fontfamily=REPORT_FONT,
        )
        cover_axis = cover.add_axes([0.07, 0.12, 0.86, 0.50])
        cover_axis.axis("off")
        cover_rows = [
            [str(row.item), format_report_value(row.value)]
            for row in RUN_SUMMARY_TABLE.itertuples(index=False)
        ]
        cover_table = cover_axis.table(
            cellText=cover_rows,
            colLabels=["Run item", "Value"],
            cellLoc="left",
            colLoc="left",
            bbox=[0, 0, 1, 1],
        )
        cover_table.auto_set_font_size(False)
        cover_table.set_fontsize(8.5)
        for (row_index, _), cell in cover_table.get_celld().items():
            cell.set_edgecolor(REPORT_COLORS["line"])
            cell.get_text().set_fontfamily(REPORT_FONT)
            if row_index == 0:
                cell.set_facecolor(REPORT_COLORS["navy"])
                cell.get_text().set_color("white")
                cell.get_text().set_fontweight("bold")
            else:
                cell.set_facecolor("white" if row_index % 2 else "#edf2fa")
                cell.get_text().set_color(REPORT_COLORS["ink"])
        cover.text(
            0.07,
            0.045,
            f"Report ID: {REPORT_RUN_ID} | Runtime profile: {PLATFORM}",
            color=REPORT_COLORS["muted"],
            fontsize=8,
            fontfamily=REPORT_FONT,
        )
        save_report_page(pdf, cover)
        page_count += 1

        audit_summary_rows = []
        if DATASET_AUDIT is not None:
            for key, value in DATASET_AUDIT["summary"].items():
                audit_summary_rows.append({
                    "audit_measure": key.replace("_", " ").title(),
                    "value": value,
                })
            audit_summary_rows.extend([
                {
                    "audit_measure": "Training Imbalance Ratio",
                    "value": DATASET_AUDIT["training_imbalance_ratio"],
                },
                {
                    "audit_measure": "Audit Duration Seconds",
                    "value": DATASET_AUDIT["duration_seconds"],
                },
            ])
        else:
            audit_summary_rows.append({
                "audit_measure": "Audit status",
                "value": "Disabled for this run",
            })
        page_count += add_dataframe_pages(
            pdf,
            pd.DataFrame(audit_summary_rows),
            "Dataset audit summary",
        )
        page_count += add_dataframe_pages(
            pdf,
            DATASET_DISTRIBUTION_TABLE,
            "Dataset class distribution",
        )

        curves = plt.figure(figsize=REPORT_PAGE_SIZE)
        add_report_frame(
            curves,
            "Training history",
            f"{len(TRAINING_HISTORY_TABLE)} epoch(s)",
        )
        axes = curves.subplots(3, 1)
        curves.subplots_adjust(left=0.11, right=0.94, top=0.86, bottom=0.09, hspace=0.42)
        plot_specs = [
            ("train_loss", "validation_loss", "Loss"),
            ("train_accuracy", "validation_accuracy", "Accuracy"),
            ("train_macro_f1", "validation_macro_f1", "Macro-F1"),
        ]
        for axis, (train_column, validation_column, label) in zip(axes, plot_specs):
            axis.plot(
                TRAINING_HISTORY_TABLE["epoch"],
                TRAINING_HISTORY_TABLE[train_column],
                color=REPORT_COLORS["blue"],
                linewidth=2,
                label="Train",
            )
            axis.plot(
                TRAINING_HISTORY_TABLE["epoch"],
                TRAINING_HISTORY_TABLE[validation_column],
                color=REPORT_COLORS["green"],
                linewidth=2,
                label="Validation",
            )
            axis.set_title(label, loc="left", fontweight="bold", fontfamily=REPORT_FONT)
            axis.set_xlabel("Epoch")
            axis.legend(frameon=False, ncol=2)
            axis.grid(alpha=0.22)
        save_report_page(pdf, curves)
        page_count += 1

        performance = plt.figure(figsize=REPORT_PAGE_SIZE)
        add_report_frame(
            performance,
            "Final test performance",
            (
                f"Accuracy {test_metrics['accuracy']:.4f} "
                f"({test_metrics['accuracy']:.2%}) | "
                f"Macro-F1 {test_metrics['macro_f1']:.4f} | "
                f"Loss {test_metrics['loss']:.4f}"
            ),
        )
        matrix_axis = performance.add_axes([0.11, 0.44, 0.78, 0.42])
        confusion_matrix = np.asarray(test_metrics["confusion"])
        matrix_image = matrix_axis.imshow(confusion_matrix, cmap="Blues")
        annotate_confusion_matrix(matrix_axis, confusion_matrix)
        matrix_axis.set_title(
            "Confusion matrix (image counts)",
            loc="left",
            fontweight="bold",
        )
        matrix_axis.set_xlabel("Predicted class")
        matrix_axis.set_ylabel("True class")
        matrix_axis.set_xticks(range(NUM_CLASSES), CLASSES, rotation=90, fontsize=6.5)
        matrix_axis.set_yticks(range(NUM_CLASSES), CLASSES, fontsize=6.5)
        performance.colorbar(matrix_image, ax=matrix_axis, fraction=0.035, pad=0.03)

        f1_axis = performance.add_axes([0.11, 0.09, 0.78, 0.25])
        positions = np.arange(NUM_CLASSES)
        f1_axis.bar(
            positions,
            TEST_CLASS_METRICS_TABLE["test_f1"],
            color=REPORT_COLORS["blue"],
        )
        f1_axis.set_ylim(0, 1)
        f1_axis.set_ylabel("F1")
        f1_axis.set_title("Per-class test F1", loc="left", fontweight="bold")
        f1_axis.set_xticks(positions, CLASSES, rotation=60, ha="right", fontsize=6.5)
        f1_axis.grid(axis="y", alpha=0.22)
        save_report_page(pdf, performance)
        page_count += 1

        page_count += add_dataframe_pages(
            pdf,
            TEST_CLASS_METRICS_TABLE,
            "Per-class test metrics",
        )
        page_count += add_dataframe_pages(
            pdf,
            TEST_CLASS_ERROR_TABLE,
            "Class-wise test error analysis",
        )
        page_count += add_dataframe_pages(
            pdf,
            TEST_CONFUSION_PAIRS_TABLE.head(30),
            "Most frequent test-set confusions",
        )

        configuration = pd.DataFrame([
            {"setting": "Resolved image size", "value": IMAGE_SIZE},
            {"setting": "Training batch size", "value": BATCH_SIZE},
            {
                "setting": "Evaluation batch size",
                "value": BATCH_SIZE * EVAL_BATCH_MULTIPLIER,
            },
            {"setting": "Workers", "value": NUM_WORKERS},
            {"setting": "Prefetch factor", "value": PREFETCH_FACTOR},
            {"setting": "Fused AdamW", "value": FUSED_ADAMW_ENABLED},
            {"setting": "TensorBoard", "value": TENSORBOARD_ENABLED},
            {
                "setting": "TensorBoard flush seconds",
                "value": TENSORBOARD_FLUSH_SECONDS,
            },
            {
                "setting": "TensorBoard histogram interval",
                "value": TENSORBOARD_HISTOGRAM_INTERVAL,
            },
            {
                "setting": "TensorBoard profiler batches",
                "value": TENSORBOARD_PROFILE_BATCHES,
            },
            {
                "setting": "TensorBoard graph",
                "value": TENSORBOARD_LOG_GRAPH,
            },
            {
                "setting": "TensorBoard PR curves",
                "value": TENSORBOARD_LOG_PR_CURVES,
            },
            {
                "setting": "TensorBoard embeddings",
                "value": TENSORBOARD_LOG_EMBEDDINGS,
            },
            {
                "setting": "TensorBoard activation interval",
                "value": TENSORBOARD_ACTIVATION_INTERVAL,
            },
            {
                "setting": "TensorBoard optimizer interval",
                "value": TENSORBOARD_OPTIMIZER_INTERVAL,
            },
            {
                "setting": "TensorBoard class metrics",
                "value": TENSORBOARD_LOG_CLASS_METRICS,
            },
            {
                "setting": "TensorBoard system metrics",
                "value": TENSORBOARD_LOG_SYSTEM_METRICS,
            },
            {
                "setting": "TensorBoard ROC and calibration",
                "value": (
                    TENSORBOARD_LOG_ROC_CURVES
                    and TENSORBOARD_LOG_CALIBRATION
                ),
            },
            {
                "setting": "TensorBoard Grad-CAM",
                "value": TENSORBOARD_LOG_GRADCAM,
            },
            {
                "setting": "TensorBoard full model histograms",
                "value": TENSORBOARD_LOG_FULL_MODEL_HISTOGRAMS,
            },
            {
                "setting": "Gap adaptation threshold",
                "value": f"{GENERALIZATION_GAP_THRESHOLD:.2%}",
            },
            {
                "setting": "Gap adaptation patience",
                "value": GAP_ADAPTATION_PATIENCE,
            },
            {
                "setting": "Regularization adaptations",
                "value": len(history["adaptations"]),
            },
            {"setting": "Backbone learning rate", "value": BACKBONE_LR},
            {"setting": "Fusion learning rate", "value": FUSION_LR},
            {"setting": "Weight decay", "value": WEIGHT_DECAY},
            {"setting": "Label smoothing", "value": LABEL_SMOOTHING},
            {"setting": "Fusion channels", "value": FUSION_CHANNELS},
            {"setting": "Attention heads", "value": ATTENTION_HEADS},
            {"setting": "Fusion depth", "value": FUSION_DEPTH},
            {"setting": "Dropout", "value": DROPOUT},
            {"setting": "Mixed precision", "value": AMP_ENABLED},
            {"setting": "Balanced sampler", "value": use_balanced_sampler},
            {
                "setting": "Balance strategy",
                "value": BALANCED_SAMPLER_STRATEGY if use_balanced_sampler else "none",
            },
            {"setting": "Effective-number beta", "value": resolved_sampler_beta},
            {"setting": "Loss function", "value": RESOLVED_LOSS_FUNCTION},
            {
                "setting": "Focal gamma",
                "value": FOCAL_GAMMA if use_focal_loss else "not used",
            },
            {
                "setting": "Minority augmentation classes",
                "value": ", ".join(
                    CLASSES[index] for index in sorted(MINORITY_CLASS_INDICES)
                ) or "none",
            },
            {"setting": "Seed", "value": SEED},
        ])
        page_count += add_dataframe_pages(pdf, configuration, "Training configuration")

        def short_artifact_path(path):
            path = Path(path)
            try:
                return path.relative_to(PROJECT_OUTPUT_ROOT).as_posix()
            except ValueError:
                return str(path)

        artifact_table = pd.DataFrame([
            {"artifact": "Best model", "path": short_artifact_path(best_checkpoint_path)},
            {
                "artifact": "Latest checkpoint",
                "path": short_artifact_path(last_checkpoint_path),
            },
            {
                "artifact": "Dataset audit",
                "path": short_artifact_path(RESULTS_DIR / "dataset_audit.json"),
            },
            {
                "artifact": "Final metrics",
                "path": short_artifact_path(RESULTS_DIR / "final_metrics.json"),
            },
            {
                "artifact": "Training curves",
                "path": short_artifact_path(RESULTS_DIR / "training_curves.png"),
            },
            {
                "artifact": "TensorBoard run",
                "path": (
                    short_artifact_path(TENSORBOARD_RUN_DIR)
                    if TENSORBOARD_RUN_DIR is not None
                    else "Disabled"
                ),
            },
            {
                "artifact": "Test confusion matrix",
                "path": short_artifact_path(RESULTS_DIR / "test_confusion_matrix.png"),
            },
            *[
                {
                    "artifact": f"CSV - {name.replace('_', ' ')}",
                    "path": short_artifact_path(path),
                }
                for name, path in CSV_ARTIFACTS.items()
            ],
        ])
        page_count += add_dataframe_pages(pdf, artifact_table, "Saved run artifacts", 18)
    if page_count != _REPORT_PAGE_NUMBER:
        raise RuntimeError("PDF page counter did not match the generated page count.")
    return page_count


PDF_REPORT_PATH = REPORTS_DIR / (
    f"ugvnet_training_report_{PLATFORM}_{REPORT_RUN_ID}.pdf"
)
PDF_PAGE_COUNT = build_training_pdf(PDF_REPORT_PATH)
LATEST_PDF_REPORT_PATH = REPORTS_DIR / "ugvnet_training_report_latest.pdf"
shutil.copy2(PDF_REPORT_PATH, LATEST_PDF_REPORT_PATH)

with PDF_REPORT_PATH.open("rb") as report_file:
    pdf_signature = report_file.read(5)
if pdf_signature != b"%PDF-" or PDF_REPORT_PATH.stat().st_size < 10_000:
    raise RuntimeError("The training report did not pass the PDF integrity check.")

report_manifest = {
    "report_id": REPORT_RUN_ID,
    "platform": PLATFORM,
    "pdf_path": str(PDF_REPORT_PATH),
    "latest_pdf_path": str(LATEST_PDF_REPORT_PATH),
    "pdf_pages": PDF_PAGE_COUNT,
    "csv_artifacts": {name: str(path) for name, path in CSV_ARTIFACTS.items()},
    "best_model": str(best_checkpoint_path),
    "latest_checkpoint": str(last_checkpoint_path),
    "tensorboard_run_dir": (
        str(TENSORBOARD_RUN_DIR) if TENSORBOARD_RUN_DIR is not None else None
    ),
}
(REPORTS_DIR / f"report_manifest_{REPORT_RUN_ID}.json").write_text(
    json.dumps(report_manifest, indent=2),
    encoding="utf-8",
)

if TENSORBOARD_WRITER is not None and not TENSORBOARD_WRITER_CLOSED:
    tensorboard_text("Artifacts/pdf_report", str(PDF_REPORT_PATH), completed_epochs)
    TENSORBOARD_WRITER.flush()
    TENSORBOARD_WRITER.close()
    TENSORBOARD_WRITER_CLOSED = True

display(HTML(
    '<div class="ugvnet-note">'
    '<strong>Training report created successfully.</strong><br>'
    f'PDF pages: {PDF_PAGE_COUNT}<br>'
    f'PDF: {PDF_REPORT_PATH}<br>'
    f'Latest copy: {LATEST_PDF_REPORT_PATH}<br>'
    f'CSV folder: {RESULTS_DIR}<br>'
    f'TensorBoard run: {TENSORBOARD_RUN_DIR}'
    '</div>'
))

## 21. Predict one image

**This cell covers:** A reusable prediction function that preprocesses one image and returns its top classes with probabilities. Change `EXAMPLE_IMAGE` before running the prediction line.


In [ ]:
# 21. Single-image prediction
def predict_image(image_path, top_k=3):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    tensor = evaluation_transform(image).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        probabilities = model(tensor).softmax(dim=1)[0]
    values, indices = probabilities.topk(min(top_k, NUM_CLASSES))
    predictions = [
        {"class": CLASSES[index], "probability": float(value)}
        for value, index in zip(values.cpu(), indices.cpu())
    ]
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="#fafbff")
    ax.set_facecolor("#fafbff")
    ax.imshow(image)
    ax.axis("off")
    title_lines = "\n".join(
        f"{p['class']}: {p['probability']:.2%}" for p in predictions
    )
    ax.set_title(
        title_lines,
        fontsize=12, fontfamily="Inter", fontweight="600",
        color="#1e293b", pad=12,
    )
    fig.tight_layout()
    plt.show()
    return predictions

# EXAMPLE_IMAGE = DATA_DIR / "test" / CLASSES[0] / "your_image.jpg"
# predict_image(EXAMPLE_IMAGE, top_k=3)


## 22. Resume from a checkpoint

**This cell covers:** The preferred full-state continuation workflow for Kaggle.

Set `RESUME_CHECKPOINT_PATH` in Cell 04, then run through Cell 13. The checkpoint is validated against the current classes and model dimensions before model, optimizer, scheduler, scaler, history, early-stopping, adaptive-regularization, freeze, and random-number states are restored. Run Cell 17 to continue.


In [ ]:
# 22. Resume training
# Preferred workflow: set RESUME_CHECKPOINT_PATH in Cell 04 before Cell 13.
# Cell 13 validates and restores the model, optimizer, scheduler, scaler,
# epoch history, early-stopping state, adaptive regularization,
# backbone freeze state, and random-number generators automatically.
#
# RESUME_CHECKPOINT_PATH = "/kaggle/input/your-checkpoint/ugvnet_hybrid_last.pt"
# Rerun Cell 13, then run Cell 17 to continue training.


## 23. Practical notes

**This cell covers:** Practical recommendations for reliable small- and large-dataset experiments in Kaggle.

- Keep patient identities separated across all splits and remove duplicates before training.
- Use validation results for model decisions; inspect the untouched test set only after training.
- For small datasets, keep pretrained weights, frozen warm-up, augmentation, early stopping, and macro-F1 selection enabled.
- In automatic mode, training splits with at least 8,000 images use the faster 224px/256-channel/one-block profile and a two-epoch frozen-backbone warm-up.
- If CUDA runs out of memory, reduce `BATCH_SIZE` first. Change the profile-specific image size or fusion width only if necessary.
- Results, models, checkpoints, reports, and caches are written directly under `/kaggle/working/ugvnet` without a redundant `kaggle/` suffix.
- Save a notebook version or download the output artifacts before ending the session.
- No epoch is rejected for a train/validation accuracy gap. A sustained gap only adjusts the following epoch's learning rate, dropout, label smoothing, and optional focal gamma.
- Effective-number sampling and minority augmentation activate from training counts only; focal loss activates automatically only at severe imbalance.
- Resume only from a trusted UGVNet checkpoint and keep its dataset class folders unchanged.
- Cell 19 writes styled CSV views and a timestamped PDF report to `REPORTS_DIR` after evaluation.
- Advanced TensorBoard logging is sampled: activations, optimizer states, and validation embeddings run on epoch 1 and then at their configured intervals.
- `TENSORBOARD_LOG_FULL_MODEL_HISTOGRAMS` is intentionally off by default because exhaustive dual-backbone histograms can slow a Kaggle run; enable it only for a diagnostic experiment.
